In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Current notebook project folder: nci_almanac_v3
PROJECT_DIR = Path(".").resolve()

# Source data folder from old project
SOURCE_DATA_DIR = (PROJECT_DIR / "../nci_almanac_v2/data").resolve()

# New output folders inside v3
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"

DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("✅ Project setup ready")
print("Project folder      :", PROJECT_DIR)
print("Source data folder  :", SOURCE_DATA_DIR)
print("New data folder     :", DATA_DIR)
print("Results folder      :", RESULTS_DIR)
print("Models folder       :", MODELS_DIR)

print("\nFiles available in source data folder:")
for file in SOURCE_DATA_DIR.iterdir():
    print(" -", file.name)

✅ Project setup ready
Project folder      : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3
Source data folder  : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data
New data folder     : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data
Results folder      : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results
Models folder       : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\models

Files available in source data folder:
 - cells_aug.npy
 - cells_raw.npy
 - ComboCompoundSet.sdf
 - ComboDrugGrowth_Nov2017.csv
 - comboscore.csv
 - drug_features.pkl
 - drug_mols.pkl
 - X_aug.npy
 - X_raw.npy
 - y_aug.npy
 - y_raw.npy


In [2]:
# ============================================================
# STEP 1: Compute corrected FG-only ComboScore
# ============================================================

import pandas as pd
import numpy as np

# Input raw file from v2/data
raw_file = SOURCE_DATA_DIR / "ComboDrugGrowth_Nov2017.csv"

# Output processed file to v3/data
output_file = DATA_DIR / "comboscore.csv"

print("Reading raw file from:")
print(raw_file)

print("\nProcessed ComboScore will be saved to:")
print(output_file)

# ------------------------------------------------------------
# 1. Load only required columns
# ------------------------------------------------------------

required_columns = [
    "SCREENER",
    "VALID",
    "NSC1",
    "NSC2",
    "CELLNAME",
    "SCORE"
]

df_raw = pd.read_csv(
    raw_file,
    usecols=required_columns,
    low_memory=False
)

print("\n✅ Raw data loaded")
print("Raw shape:", df_raw.shape)

print("\nSCREENER distribution before filtering:")
print(df_raw["SCREENER"].value_counts(dropna=False))

print("\nVALID distribution before filtering:")
print(df_raw["VALID"].value_counts(dropna=False))


# ------------------------------------------------------------
# 2. Clean important columns
# ------------------------------------------------------------

df_raw["SCREENER"] = df_raw["SCREENER"].astype(str).str.strip().str.upper()
df_raw["VALID"] = df_raw["VALID"].astype(str).str.strip().str.upper()

df_raw["NSC1"] = pd.to_numeric(df_raw["NSC1"], errors="coerce")
df_raw["NSC2"] = pd.to_numeric(df_raw["NSC2"], errors="coerce")
df_raw["SCORE"] = pd.to_numeric(df_raw["SCORE"], errors="coerce")


# ------------------------------------------------------------
# 3. Filter only valid FG data
# ------------------------------------------------------------

df_fg = df_raw[
    (df_raw["VALID"] == "Y") &
    (df_raw["SCREENER"] == "FG")
].copy()

print("\n✅ FG-only valid rows selected")
print("FG valid shape before dropping missing values:", df_fg.shape)

# Remove unusable rows
df_fg = df_fg.dropna(subset=["NSC1", "NSC2", "CELLNAME", "SCORE"]).copy()

df_fg["NSC1"] = df_fg["NSC1"].astype(int)
df_fg["NSC2"] = df_fg["NSC2"].astype(int)

print("FG valid shape after dropping missing values:", df_fg.shape)


# ------------------------------------------------------------
# 4. Make drug pair order consistent
#    Example: drug A + drug B is same as drug B + drug A
# ------------------------------------------------------------

df_fg["DRUG_A"] = df_fg[["NSC1", "NSC2"]].min(axis=1)
df_fg["DRUG_B"] = df_fg[["NSC1", "NSC2"]].max(axis=1)


# ------------------------------------------------------------
# 5. Compute ComboScore
#    Correct formula:
#    For each drug pair and cell line:
#    COMBOSCORE = sum(SCORE)
# ------------------------------------------------------------

comboscore = (
    df_fg
    .groupby(["DRUG_A", "DRUG_B", "CELLNAME"], as_index=False)["SCORE"]
    .sum()
    .rename(columns={
        "DRUG_A": "NSC1",
        "DRUG_B": "NSC2",
        "SCORE": "COMBOSCORE"
    })
)

comboscore = comboscore.sort_values(
    ["CELLNAME", "NSC1", "NSC2"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 6. Validate final ComboScore data
# ------------------------------------------------------------

print("\n============================================================")
print("FG-only ComboScore validation")
print("============================================================")

print("Final shape:", comboscore.shape)

print("\nColumns:")
print(comboscore.columns.tolist())

print("\nMissing values:")
print(comboscore.isna().sum())

print("\nUnique cell lines:", comboscore["CELLNAME"].nunique())

unique_drugs = pd.concat([comboscore["NSC1"], comboscore["NSC2"]]).nunique()
print("Unique drugs:", unique_drugs)

unique_pairs = comboscore[["NSC1", "NSC2"]].drop_duplicates().shape[0]
print("Unique drug pairs:", unique_pairs)

print("\nComboScore statistics:")
print(comboscore["COMBOSCORE"].describe())

print("\nSynergy distribution:")
print("Synergistic  (> 0):", (comboscore["COMBOSCORE"] > 0).sum())
print("Antagonistic (< 0):", (comboscore["COMBOSCORE"] < 0).sum())
print("Neutral      (= 0):", (comboscore["COMBOSCORE"] == 0).sum())

print("\nRows per cell line:")
print(comboscore["CELLNAME"].value_counts().sort_index())


# ------------------------------------------------------------
# 7. Save processed data into v3/data
# ------------------------------------------------------------

comboscore.to_csv(output_file, index=False)

print("\n✅ Corrected FG-only ComboScore saved successfully")
print("Saved file:", output_file)

print("\nPreview:")
display(comboscore.head())

Reading raw file from:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\ComboDrugGrowth_Nov2017.csv

Processed ComboScore will be saved to:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\comboscore.csv

✅ Raw data loaded
Raw shape: (3686475, 6)

SCREENER distribution before filtering:
SCREENER
FF    2062098
FG    1415772
1A     208605
Name: count, dtype: int64

VALID distribution before filtering:
VALID
Y    3686475
Name: count, dtype: int64

✅ FG-only valid rows selected
FG valid shape before dropping missing values: (1415772, 6)
FG valid shape after dropping missing values: (1346094, 6)

FG-only ComboScore validation
Final shape: (145212, 4)

Columns:
['NSC1', 'NSC2', 'CELLNAME', 'COMBOSCORE']

Missing values:
NSC1          0
NSC2          0
CELLNAME      0
COMBOSCORE    0
dtype: int64

Unique cell lines: 60
Unique drugs: 100
Unique drug pairs: 2446

ComboScore statistics:
count    145212.000000
mean        -16.742266
std          59.782866
min       -1148.000000
25%       

,NSC1,NSC2,CELLNAME,COMBOSCORE
0,740,752,786-0,-80.0
1,740,3088,786-0,-44.0
2,740,8806,786-0,-64.0
3,740,13875,786-0,-14.0
4,740,19893,786-0,-54.0


In [5]:
print(DATA_DIR)
print(SOURCE_DATA_DIR)
print(RESULTS_DIR)

C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results


In [6]:
# ============================================================
# STEP 2: Load, clean, and validate drug features
# ============================================================

import json
import pickle
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

comboscore_file = DATA_DIR / "comboscore.csv"

source_features_pkl = SOURCE_DATA_DIR / "drug_features.pkl"

out_features_csv = DATA_DIR / "drug_features.csv"
out_features_pkl = DATA_DIR / "drug_features.pkl"
out_feature_cols_json = DATA_DIR / "feature_columns.json"
out_coverage_report = RESULTS_DIR / "step2_drug_feature_coverage_report.csv"

print("Reading corrected ComboScore from:")
print(comboscore_file)

print("\nReading source drug features from:")
print(source_features_pkl)

print("\nClean drug features will be saved to:")
print(out_features_csv)


# ------------------------------------------------------------
# 1. Check required input files
# ------------------------------------------------------------

if not comboscore_file.exists():
    raise FileNotFoundError(
        f"Missing {comboscore_file}. Run Step 1 first and create comboscore.csv."
    )

if not source_features_pkl.exists():
    raise FileNotFoundError(
        f"Missing {source_features_pkl}. Cannot continue Step 2."
    )


# ------------------------------------------------------------
# 2. Load corrected FG-only ComboScore
# ------------------------------------------------------------

comboscore = pd.read_csv(comboscore_file)

required_combo_cols = ["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]
missing_combo_cols = [c for c in required_combo_cols if c not in comboscore.columns]

if missing_combo_cols:
    raise ValueError(f"comboscore.csv is missing required columns: {missing_combo_cols}")

comboscore["NSC1"] = pd.to_numeric(comboscore["NSC1"], errors="coerce")
comboscore["NSC2"] = pd.to_numeric(comboscore["NSC2"], errors="coerce")
comboscore["COMBOSCORE"] = pd.to_numeric(comboscore["COMBOSCORE"], errors="coerce")

before_drop = len(comboscore)
comboscore = comboscore.dropna(subset=["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]).copy()
after_drop = len(comboscore)

comboscore["NSC1"] = comboscore["NSC1"].astype(int)
comboscore["NSC2"] = comboscore["NSC2"].astype(int)

print("\n✅ ComboScore loaded")
print("ComboScore shape:", comboscore.shape)
print("Rows dropped because of missing required values:", before_drop - after_drop)
print("Unique cell lines:", comboscore["CELLNAME"].nunique())

needed_nscs = sorted(
    set(comboscore["NSC1"].unique()).union(set(comboscore["NSC2"].unique()))
)

print("Unique drugs needed by FG-only ComboScore:", len(needed_nscs))


# ------------------------------------------------------------
# 3. Load drug_features.pkl safely
# ------------------------------------------------------------

with open(source_features_pkl, "rb") as f:
    feature_obj = pickle.load(f)

print("\n✅ drug_features.pkl loaded")
print("Object type:", type(feature_obj))


# ------------------------------------------------------------
# 4. Convert loaded object into a DataFrame
# ------------------------------------------------------------

def dict_to_feature_dataframe(obj):
    """
    Converts common dictionary formats to DataFrame.

    Expected common format:
        {
            NSC_ID_1: feature_vector,
            NSC_ID_2: feature_vector,
            ...
        }

    feature_vector can be list, numpy array, pandas Series, or dict.
    """
    rows = []

    for key, value in obj.items():
        row = {"NSC": key}

        if isinstance(value, dict):
            for k, v in value.items():
                row[str(k)] = v

        elif isinstance(value, pd.Series):
            for k, v in value.to_dict().items():
                row[str(k)] = v

        else:
            arr = np.asarray(value).ravel()
            for i, v in enumerate(arr):
                row[f"feat_{i}"] = v

        rows.append(row)

    return pd.DataFrame(rows)


if isinstance(feature_obj, pd.DataFrame):
    features = feature_obj.copy()

elif isinstance(feature_obj, dict):
    # Case 1: dictionary directly maps NSC -> feature vector
    features = dict_to_feature_dataframe(feature_obj)

elif isinstance(feature_obj, np.ndarray):
    raise ValueError(
        "drug_features.pkl is a NumPy array without NSC IDs. "
        "We cannot match features to NSC1/NSC2 safely. "
        "Use a feature file that contains NSC identifiers."
    )

else:
    raise TypeError(
        f"Unsupported drug_features.pkl format: {type(feature_obj)}. "
        "Expected DataFrame or dictionary."
    )

print("\nConverted features shape:", features.shape)
print("First columns:", features.columns[:10].tolist())


# ------------------------------------------------------------
# 5. Find or create NSC column
# ------------------------------------------------------------

possible_nsc_cols = [
    c for c in features.columns
    if str(c).strip().upper() in ["NSC", "NSC_ID", "NSC1", "ID"]
]

if len(possible_nsc_cols) > 0:
    nsc_col = possible_nsc_cols[0]
    features = features.rename(columns={nsc_col: "NSC"})

else:
    # Sometimes NSC is stored as index
    index_as_series = pd.Series(features.index)

    numeric_index = pd.to_numeric(index_as_series, errors="coerce")

    if numeric_index.notna().mean() > 0.95:
        features = features.reset_index().rename(columns={"index": "NSC"})
    else:
        raise ValueError(
            "Could not find NSC column in drug_features.pkl. "
            "Feature data must contain NSC IDs so we can match drug features."
        )

features["NSC"] = pd.to_numeric(features["NSC"], errors="coerce")

before_nsc_drop = len(features)
features = features.dropna(subset=["NSC"]).copy()
after_nsc_drop = len(features)

features["NSC"] = features["NSC"].astype(int)

print("\n✅ NSC column prepared")
print("Rows dropped because NSC was missing/invalid:", before_nsc_drop - after_nsc_drop)
print("Feature rows after NSC cleaning:", features.shape)


# ------------------------------------------------------------
# 6. Keep only numeric feature columns
# ------------------------------------------------------------

# Remove duplicate non-feature identifier-like columns if present
identifier_like_cols = {
    "NSC",
    "SMILES",
    "smiles",
    "NAME",
    "Name",
    "name",
    "DRUG",
    "Drug",
    "drug",
    "MOL",
    "mol",
    "ROMol",
}

candidate_feature_cols = [
    c for c in features.columns
    if c not in identifier_like_cols
]

numeric_feature_cols = []

for col in candidate_feature_cols:
    converted = pd.to_numeric(features[col], errors="coerce")

    # Keep column only if at least some numeric values exist
    if converted.notna().sum() > 0:
        features[col] = converted
        numeric_feature_cols.append(col)

if len(numeric_feature_cols) == 0:
    raise ValueError("No numeric feature columns found in drug_features.pkl.")

features = features[["NSC"] + numeric_feature_cols].copy()

print("\n✅ Numeric feature columns selected")
print("Number of numeric feature columns:", len(numeric_feature_cols))


# ------------------------------------------------------------
# 7. Handle duplicates and missing feature values
# ------------------------------------------------------------

duplicate_nsc_count = features["NSC"].duplicated().sum()

if duplicate_nsc_count > 0:
    print("\nDuplicate NSC rows found:", duplicate_nsc_count)
    print("Averaging duplicate feature rows by NSC...")
    features = features.groupby("NSC", as_index=False)[numeric_feature_cols].mean()
else:
    print("\nNo duplicate NSC rows found.")

missing_before = features[numeric_feature_cols].isna().sum().sum()

if missing_before > 0:
    print("Missing numeric feature values before filling:", missing_before)
    medians = features[numeric_feature_cols].median(numeric_only=True)
    features[numeric_feature_cols] = features[numeric_feature_cols].fillna(medians)

missing_after = features[numeric_feature_cols].isna().sum().sum()

print("Missing numeric feature values after filling:", missing_after)


# ------------------------------------------------------------
# 8. Remove unusable feature columns
# ------------------------------------------------------------

# Remove columns that are still fully missing, if any
all_missing_cols = [
    c for c in numeric_feature_cols
    if features[c].isna().all()
]

if all_missing_cols:
    print("\nRemoving fully missing feature columns:", len(all_missing_cols))
    features = features.drop(columns=all_missing_cols)
    numeric_feature_cols = [c for c in numeric_feature_cols if c not in all_missing_cols]

# Remove columns with only one value because they cannot help model training
constant_cols = [
    c for c in numeric_feature_cols
    if features[c].nunique(dropna=True) <= 1
]

if constant_cols:
    print("Removing constant feature columns:", len(constant_cols))
    features = features.drop(columns=constant_cols)
    numeric_feature_cols = [c for c in numeric_feature_cols if c not in constant_cols]

print("\nFeature columns after cleanup:", len(numeric_feature_cols))


# ------------------------------------------------------------
# 9. Validate against paper-style expectation
# ------------------------------------------------------------

print("\n============================================================")
print("Paper-style feature check")
print("============================================================")

print("Expected best RF feature setup from paper:")
print("MFPC count features: 256")
print("Physico-chemical features: 7")
print("Expected total per drug: 263")

actual_feature_count = len(numeric_feature_cols)

print("\nActual numeric feature columns found:", actual_feature_count)

if actual_feature_count == 263:
    print("✅ Feature count matches paper-style MFPC + physico-chemical setup.")
else:
    print("⚠️ Feature count is not exactly 263.")
    print("   This may still work, but it may not exactly match the paper setup.")
    print("   Check whether your old drug_features.pkl was generated with MFPC + 7 descriptors.")


# ------------------------------------------------------------
# 10. Validate coverage with corrected FG-only ComboScore
# ------------------------------------------------------------

available_nscs = set(features["NSC"].unique())
needed_nscs_set = set(needed_nscs)

missing_nscs = sorted(needed_nscs_set - available_nscs)
extra_nscs = sorted(available_nscs - needed_nscs_set)

coverage = 100 * (len(needed_nscs_set) - len(missing_nscs)) / len(needed_nscs_set)

print("\n============================================================")
print("Coverage check against corrected FG-only ComboScore")
print("============================================================")

print("Drugs needed by comboscore.csv:", len(needed_nscs_set))
print("Drugs available in features:", len(available_nscs))
print("Missing needed drugs:", len(missing_nscs))
print(f"Coverage: {coverage:.2f}%")

if len(missing_nscs) > 0:
    print("\nFirst 20 missing NSCs:")
    print(missing_nscs[:20])
else:
    print("✅ All drugs in corrected ComboScore have feature vectors.")

coverage_report = pd.DataFrame({
    "NSC": needed_nscs,
    "has_features": [nsc in available_nscs for nsc in needed_nscs]
})

coverage_report.to_csv(out_coverage_report, index=False)


# ------------------------------------------------------------
# 11. Keep only drugs used in corrected ComboScore
# ------------------------------------------------------------

features_used = features[features["NSC"].isin(needed_nscs_set)].copy()
features_used = features_used.sort_values("NSC").reset_index(drop=True)

print("\nFinal cleaned feature table shape:", features_used.shape)


# ------------------------------------------------------------
# 12. Save clean feature files into v3/data
# ------------------------------------------------------------

features_used.to_csv(out_features_csv, index=False)
features_used.to_pickle(out_features_pkl)

with open(out_feature_cols_json, "w") as f:
    json.dump(numeric_feature_cols, f, indent=2)

print("\n✅ Clean drug features saved successfully")
print("CSV saved to:", out_features_csv)
print("PKL saved to:", out_features_pkl)
print("Feature columns saved to:", out_feature_cols_json)
print("Coverage report saved to:", out_coverage_report)

print("\nPreview:")
display(features_used.head())

print("\n✅ STEP 2 COMPLETE")

Reading corrected ComboScore from:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\comboscore.csv

Reading source drug features from:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\drug_features.pkl

Clean drug features will be saved to:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv

✅ ComboScore loaded
ComboScore shape: (145212, 4)
Rows dropped because of missing required values: 0
Unique cell lines: 60
Unique drugs needed by FG-only ComboScore: 100

✅ drug_features.pkl loaded
Object type: <class 'dict'>

Converted features shape: (103, 264)
First columns: ['NSC', 'feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8']

✅ NSC column prepared
Rows dropped because NSC was missing/invalid: 0
Feature rows after NSC cleaning: (103, 264)

✅ Numeric feature columns selected
Number of numeric feature columns: 263

No duplicate NSC rows found.
Missing numeric feature values after filling: 0

Feature columns after cleanup: 26

,NSC,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_253,feat_254,feat_255,feat_256,feat_257,feat_258,feat_259,feat_260,feat_261,feat_262
0,740,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,454.446991,0.26840,210.539993,3.0,0.0,5.0,10.0
1,750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,246.306000,-0.28100,86.739998,0.0,0.0,0.0,6.0
2,752,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,167.197006,0.59769,83.379997,2.0,0.0,3.0,4.0
3,755,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,152.182007,1.01549,57.360001,2.0,0.0,2.0,3.0
4,762,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,192.516998,1.81760,3.240000,0.0,0.0,0.0,1.0



✅ STEP 2 COMPLETE


In [7]:
# ============================================================
# STEP 2B: Investigate missing drug features
# ============================================================

import pickle
import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

comboscore_file = DATA_DIR / "comboscore.csv"
clean_features_file = DATA_DIR / "drug_features.csv"

source_sdf_file = SOURCE_DATA_DIR / "ComboCompoundSet.sdf"
source_mols_pkl = SOURCE_DATA_DIR / "drug_mols.pkl"

missing_report_file = RESULTS_DIR / "step2b_missing_drug_investigation.csv"
affected_rows_file = RESULTS_DIR / "step2b_rows_affected_by_missing_drugs.csv"

print("ComboScore file:", comboscore_file)
print("Clean features file:", clean_features_file)
print("Source SDF file:", source_sdf_file)
print("Source drug_mols.pkl:", source_mols_pkl)


# ------------------------------------------------------------
# 1. Load corrected ComboScore and clean features
# ------------------------------------------------------------

comboscore = pd.read_csv(comboscore_file)
features = pd.read_csv(clean_features_file)

comboscore["NSC1"] = pd.to_numeric(comboscore["NSC1"], errors="coerce").astype("Int64")
comboscore["NSC2"] = pd.to_numeric(comboscore["NSC2"], errors="coerce").astype("Int64")
features["NSC"] = pd.to_numeric(features["NSC"], errors="coerce").astype("Int64")

needed_nscs = set(comboscore["NSC1"].dropna().astype(int)).union(
    set(comboscore["NSC2"].dropna().astype(int))
)

available_feature_nscs = set(features["NSC"].dropna().astype(int))

missing_nscs = sorted(needed_nscs - available_feature_nscs)

print("\n============================================================")
print("Missing NSC check")
print("============================================================")
print("Drugs needed by FG-only ComboScore:", len(needed_nscs))
print("Drugs available in clean features:", len(available_feature_nscs))
print("Missing drugs:", missing_nscs)

if len(missing_nscs) == 0:
    print("\n✅ No missing drugs. Step 2B not needed further.")
else:
    print("\n⚠️ Missing drug features found. Investigating now...")


# ------------------------------------------------------------
# 2. Helper function to extract NSC number from text
# ------------------------------------------------------------

def extract_nsc_from_text(value):
    """
    Extracts an NSC-like number from text.
    Example:
        'NSC 119875' -> 119875
        '119875' -> 119875
    """
    if value is None:
        return None
    
    text = str(value)
    matches = re.findall(r"\d+", text)
    
    if not matches:
        return None
    
    # Usually NSC IDs are 3+ digit numbers
    candidates = [m for m in matches if len(m) >= 3]
    
    if not candidates:
        return None
    
    return int(candidates[0])


# ------------------------------------------------------------
# 3. Check if missing NSCs appear as text inside SDF file
# ------------------------------------------------------------

sdf_text_presence = {}

if source_sdf_file.exists():
    sdf_text = source_sdf_file.read_text(errors="ignore")
    
    for nsc in missing_nscs:
        sdf_text_presence[nsc] = str(nsc) in sdf_text
else:
    for nsc in missing_nscs:
        sdf_text_presence[nsc] = False

print("\n============================================================")
print("SDF raw text search")
print("============================================================")

for nsc in missing_nscs:
    print(f"NSC {nsc} found in SDF text:", sdf_text_presence[nsc])


# ------------------------------------------------------------
# 4. Check SDF using RDKit, if RDKit is available
# ------------------------------------------------------------

sdf_rdkit_presence = {nsc: False for nsc in missing_nscs}
sdf_rdkit_props = {nsc: "" for nsc in missing_nscs}

try:
    from rdkit import Chem
    
    if source_sdf_file.exists():
        supplier = Chem.SDMolSupplier(str(source_sdf_file), sanitize=False, removeHs=False)
        
        for mol in supplier:
            if mol is None:
                continue
            
            prop_names = list(mol.GetPropNames())
            
            possible_values = []
            
            # molecule name
            try:
                possible_values.append(mol.GetProp("_Name"))
            except Exception:
                pass
            
            # all SDF properties
            for prop in prop_names:
                try:
                    possible_values.append(mol.GetProp(prop))
                except Exception:
                    pass
            
            extracted_values = [extract_nsc_from_text(v) for v in possible_values]
            extracted_values = [v for v in extracted_values if v is not None]
            
            for nsc in missing_nscs:
                if nsc in extracted_values:
                    sdf_rdkit_presence[nsc] = True
                    sdf_rdkit_props[nsc] = ", ".join(prop_names)
    
    print("\n============================================================")
    print("SDF RDKit search")
    print("============================================================")
    
    for nsc in missing_nscs:
        print(f"NSC {nsc} found by RDKit in SDF:", sdf_rdkit_presence[nsc])

except Exception as e:
    print("\n⚠️ RDKit SDF check skipped or failed.")
    print("Reason:", e)


# ------------------------------------------------------------
# 5. Check drug_mols.pkl
# ------------------------------------------------------------

mols_pkl_presence = {nsc: False for nsc in missing_nscs}
mols_pkl_location = {nsc: "" for nsc in missing_nscs}

if source_mols_pkl.exists():
    with open(source_mols_pkl, "rb") as f:
        mols_obj = pickle.load(f)
    
    print("\n============================================================")
    print("drug_mols.pkl check")
    print("============================================================")
    print("drug_mols.pkl object type:", type(mols_obj))
    
    # Case A: dictionary format
    if isinstance(mols_obj, dict):
        normalized_keys = {}
        
        for key in mols_obj.keys():
            key_nsc = extract_nsc_from_text(key)
            if key_nsc is not None:
                normalized_keys[key_nsc] = key
        
        for nsc in missing_nscs:
            if nsc in normalized_keys:
                mols_pkl_presence[nsc] = True
                mols_pkl_location[nsc] = f"dict key: {normalized_keys[nsc]}"
        
        # Also check molecule properties if possible
        for key, mol in mols_obj.items():
            possible_values = [key]
            
            if hasattr(mol, "GetPropNames"):
                try:
                    possible_values.append(mol.GetProp("_Name"))
                except Exception:
                    pass
                
                try:
                    for prop in mol.GetPropNames():
                        possible_values.append(mol.GetProp(prop))
                except Exception:
                    pass
            
            extracted_values = [extract_nsc_from_text(v) for v in possible_values]
            extracted_values = [v for v in extracted_values if v is not None]
            
            for nsc in missing_nscs:
                if nsc in extracted_values:
                    mols_pkl_presence[nsc] = True
                    mols_pkl_location[nsc] = f"dict entry around key: {key}"
    
    # Case B: list or tuple format
    elif isinstance(mols_obj, (list, tuple)):
        for idx, mol in enumerate(mols_obj):
            possible_values = []
            
            if hasattr(mol, "GetPropNames"):
                try:
                    possible_values.append(mol.GetProp("_Name"))
                except Exception:
                    pass
                
                try:
                    for prop in mol.GetPropNames():
                        possible_values.append(mol.GetProp(prop))
                except Exception:
                    pass
            
            extracted_values = [extract_nsc_from_text(v) for v in possible_values]
            extracted_values = [v for v in extracted_values if v is not None]
            
            for nsc in missing_nscs:
                if nsc in extracted_values:
                    mols_pkl_presence[nsc] = True
                    mols_pkl_location[nsc] = f"list index: {idx}"
    
    else:
        print("Unsupported drug_mols.pkl structure for deep search.")
    
    for nsc in missing_nscs:
        print(f"NSC {nsc} found in drug_mols.pkl:", mols_pkl_presence[nsc])
        if mols_pkl_location[nsc]:
            print("  Location:", mols_pkl_location[nsc])

else:
    print("\n⚠️ drug_mols.pkl not found.")


# ------------------------------------------------------------
# 6. Count rows affected in ComboScore
# ------------------------------------------------------------

affected_rows = comboscore[
    comboscore["NSC1"].astype(int).isin(missing_nscs) |
    comboscore["NSC2"].astype(int).isin(missing_nscs)
].copy()

unaffected_rows = comboscore.drop(affected_rows.index).copy()

print("\n============================================================")
print("Affected ComboScore rows")
print("============================================================")

print("Total ComboScore rows:", len(comboscore))
print("Rows affected by missing drug features:", len(affected_rows))
print("Rows remaining if we drop affected rows:", len(unaffected_rows))
print("Percentage affected:", round(len(affected_rows) / len(comboscore) * 100, 2), "%")

print("\nAffected rows by missing NSC:")
for nsc in missing_nscs:
    count = (
        (comboscore["NSC1"].astype(int) == nsc) |
        (comboscore["NSC2"].astype(int) == nsc)
    ).sum()
    print(f"NSC {nsc}: {count} rows")

print("\nAffected rows by cell line:")
print(affected_rows["CELLNAME"].value_counts().sort_index())


# ------------------------------------------------------------
# 7. Save investigation reports
# ------------------------------------------------------------

report_rows = []

for nsc in missing_nscs:
    affected_count = (
        (comboscore["NSC1"].astype(int) == nsc) |
        (comboscore["NSC2"].astype(int) == nsc)
    ).sum()
    
    report_rows.append({
        "NSC": nsc,
        "found_in_sdf_text": sdf_text_presence.get(nsc, False),
        "found_in_sdf_rdkit": sdf_rdkit_presence.get(nsc, False),
        "found_in_drug_mols_pkl": mols_pkl_presence.get(nsc, False),
        "drug_mols_location": mols_pkl_location.get(nsc, ""),
        "affected_comboscore_rows": int(affected_count)
    })

missing_report = pd.DataFrame(report_rows)

missing_report.to_csv(missing_report_file, index=False)
affected_rows.to_csv(affected_rows_file, index=False)

print("\n✅ Missing drug investigation report saved:")
print(missing_report_file)

print("\n✅ Affected ComboScore rows saved:")
print(affected_rows_file)

print("\nInvestigation summary:")
display(missing_report)

print("\nAffected rows preview:")
display(affected_rows.head())

print("\n✅ STEP 2B COMPLETE")

ComboScore file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\comboscore.csv
Clean features file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv
Source SDF file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\ComboCompoundSet.sdf
Source drug_mols.pkl: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\drug_mols.pkl

Missing NSC check
Drugs needed by FG-only ComboScore: 100
Drugs available in clean features: 98
Missing drugs: [119875, 753082]

⚠️ Missing drug features found. Investigating now...

SDF raw text search
NSC 119875 found in SDF text: True
NSC 753082 found in SDF text: False

SDF RDKit search
NSC 119875 found by RDKit in SDF: True
NSC 753082 found by RDKit in SDF: False

drug_mols.pkl check
drug_mols.pkl object type: <class 'dict'>
NSC 119875 found in drug_mols.pkl: True
  Location: dict entry around key: 119875
NSC 753082 found in drug_mols.pkl: False

Affected ComboScore rows
Total ComboScore rows: 145212
Rows affected by missing drug

,NSC,found_in_sdf_text,found_in_sdf_rdkit,found_in_drug_mols_pkl,drug_mols_location,affected_comboscore_rows
0,119875,True,True,True,dict entry around key: 119875,1677
1,753082,False,False,False,,5847



Affected rows preview:


,NSC1,NSC2,CELLNAME,COMBOSCORE
22,740,753082,786-0,0.0
51,750,753082,786-0,-13.0
93,752,119875,786-0,-98.0
145,752,753082,786-0,19.0
175,755,753082,786-0,-57.0



✅ STEP 2B COMPLETE


In [9]:
# ============================================================
# STEP 2C FIXED: Recover missing drug features and restore coverage
# ============================================================

import json
import pickle
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

comboscore_file = DATA_DIR / "comboscore.csv"
current_features_file = DATA_DIR / "drug_features.csv"
feature_cols_json = DATA_DIR / "feature_columns.json"

source_features_pkl = SOURCE_DATA_DIR / "drug_features.pkl"
source_mols_pkl = SOURCE_DATA_DIR / "drug_mols.pkl"
source_sdf_file = SOURCE_DATA_DIR / "ComboCompoundSet.sdf"

recovery_report_file = RESULTS_DIR / "step2c_feature_recovery_report.csv"
coverage_after_file = RESULTS_DIR / "step2c_coverage_after_recovery.csv"

print("ComboScore file:", comboscore_file)
print("Current clean features:", current_features_file)
print("Source features:", source_features_pkl)
print("Source mols:", source_mols_pkl)
print("Source SDF:", source_sdf_file)


# ------------------------------------------------------------
# 1. Load current ComboScore and feature table
# ------------------------------------------------------------

comboscore = pd.read_csv(comboscore_file)
features = pd.read_csv(current_features_file)

comboscore["NSC1"] = pd.to_numeric(comboscore["NSC1"], errors="coerce").astype(int)
comboscore["NSC2"] = pd.to_numeric(comboscore["NSC2"], errors="coerce").astype(int)
features["NSC"] = pd.to_numeric(features["NSC"], errors="coerce").astype(int)

needed_nscs = set(comboscore["NSC1"].unique()).union(set(comboscore["NSC2"].unique()))
available_nscs = set(features["NSC"].unique())
missing_nscs = sorted(needed_nscs - available_nscs)

print("\nBefore recovery:")
print("Drugs needed:", len(needed_nscs))
print("Drugs available:", len(available_nscs))
print("Missing drugs:", missing_nscs)

if feature_cols_json.exists():
    with open(feature_cols_json, "r") as f:
        feature_cols = json.load(f)
else:
    feature_cols = [c for c in features.columns if c != "NSC"]

if len(feature_cols) != 263:
    raise ValueError(f"Expected 263 feature columns, found {len(feature_cols)}")

print("\nFeature columns:", len(feature_cols))


# ------------------------------------------------------------
# 2. RDKit imports
# ------------------------------------------------------------

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
except Exception as e:
    raise ImportError(
        "RDKit is required for Step 2C. Install RDKit before running this cell."
    ) from e


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def extract_nsc_from_text(value):
    if value is None:
        return None
    
    text = str(value)
    matches = re.findall(r"\d+", text)
    candidates = [m for m in matches if len(m) >= 3]
    
    if not candidates:
        return None
    
    return int(candidates[0])


def prepare_mol_for_fingerprint(mol):
    """
    Fixed RDKit molecule preparation.

    This handles molecules loaded with sanitize=False.
    It forces:
    - property cache update
    - ring info initialization
    - partial/full sanitization where possible

    This fixes:
    RuntimeError: RingInfo not initialized
    """
    if mol is None:
        return None
    
    mol = Chem.Mol(mol)
    
    # First try full sanitization
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        # If full sanitization fails, do minimum safe preparation
        try:
            mol.UpdatePropertyCache(strict=False)
        except Exception:
            pass
        
        try:
            Chem.GetSymmSSSR(mol)
        except Exception:
            try:
                Chem.FastFindRings(mol)
            except Exception:
                pass
        
        # Try sanitization again but skip operations that often fail
        try:
            sanitize_ops = (
                Chem.SanitizeFlags.SANITIZE_FINDRADICALS |
                Chem.SanitizeFlags.SANITIZE_SETAROMATICITY |
                Chem.SanitizeFlags.SANITIZE_SETCONJUGATION |
                Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION |
                Chem.SanitizeFlags.SANITIZE_SYMMRINGS
            )
            Chem.SanitizeMol(mol, sanitizeOps=sanitize_ops, catchErrors=True)
        except Exception:
            pass
    
    # Force cache and ring information again after sanitization attempt
    try:
        mol.UpdatePropertyCache(strict=False)
    except Exception:
        pass
    
    try:
        Chem.GetSymmSSSR(mol)
    except Exception:
        try:
            Chem.FastFindRings(mol)
        except Exception:
            pass
    
    try:
        mol = Chem.RemoveHs(mol, sanitize=False)
    except Exception:
        pass
    
    try:
        mol.UpdatePropertyCache(strict=False)
        Chem.GetSymmSSSR(mol)
    except Exception:
        pass
    
    return mol


def generate_263_features_from_mol(mol):
    """
    Feature order:
    feat_0   to feat_255 = Morgan count fingerprint, radius 2, 256 bits
    feat_256 = molecular weight
    feat_257 = logP
    feat_258 = TPSA
    feat_259 = H-bond donors
    feat_260 = aliphatic rings
    feat_261 = aromatic rings
    feat_262 = H-bond acceptors
    """
    mol = prepare_mol_for_fingerprint(mol)
    
    if mol is None:
        raise ValueError("Cannot generate features because molecule is None.")
    
    # Important: this needs ring info initialized
    fp = rdMolDescriptors.GetHashedMorganFingerprint(
        mol,
        radius=2,
        nBits=256
    )
    
    fp_array = np.zeros(256, dtype=float)
    
    for idx, count in fp.GetNonzeroElements().items():
        fp_array[idx] = float(count)
    
    descriptors = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        rdMolDescriptors.CalcTPSA(mol),
        Lipinski.NumHDonors(mol),
        rdMolDescriptors.CalcNumAliphaticRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        Lipinski.NumHAcceptors(mol)
    ], dtype=float)
    
    full_vector = np.concatenate([fp_array, descriptors])
    
    if len(full_vector) != 263:
        raise ValueError(f"Generated feature vector length is {len(full_vector)}, expected 263.")
    
    return full_vector


def vector_to_feature_row(nsc, vector):
    row = {"NSC": int(nsc)}
    
    for col, value in zip(feature_cols, vector):
        row[col] = float(value)
    
    return row


def dict_to_feature_dataframe(obj):
    rows = []
    
    for key, value in obj.items():
        row = {"NSC": key}
        
        if isinstance(value, dict):
            for k, v in value.items():
                row[str(k)] = v
        
        elif isinstance(value, pd.Series):
            for k, v in value.to_dict().items():
                row[str(k)] = v
        
        else:
            arr = np.asarray(value).ravel()
            for i, v in enumerate(arr):
                row[f"feat_{i}"] = v
        
        rows.append(row)
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 4. Load source feature pkl fully
# ------------------------------------------------------------

with open(source_features_pkl, "rb") as f:
    source_feature_obj = pickle.load(f)

if isinstance(source_feature_obj, pd.DataFrame):
    source_features = source_feature_obj.copy()

elif isinstance(source_feature_obj, dict):
    source_features = dict_to_feature_dataframe(source_feature_obj)

else:
    source_features = pd.DataFrame()

if not source_features.empty:
    possible_nsc_cols = [
        c for c in source_features.columns
        if str(c).strip().upper() in ["NSC", "NSC_ID", "NSC1", "ID"]
    ]
    
    if possible_nsc_cols:
        source_features = source_features.rename(columns={possible_nsc_cols[0]: "NSC"})
    
    source_features["NSC"] = pd.to_numeric(source_features["NSC"], errors="coerce")
    source_features = source_features.dropna(subset=["NSC"]).copy()
    source_features["NSC"] = source_features["NSC"].astype(int)
    
    for col in feature_cols:
        if col in source_features.columns:
            source_features[col] = pd.to_numeric(source_features[col], errors="coerce")
    
    usable_cols = ["NSC"] + [c for c in feature_cols if c in source_features.columns]
    source_features = source_features[usable_cols].copy()

print("\nSource feature table shape:", source_features.shape)


# ------------------------------------------------------------
# 5. Load local molecule dictionary
# ------------------------------------------------------------

with open(source_mols_pkl, "rb") as f:
    mols_obj = pickle.load(f)

print("drug_mols.pkl type:", type(mols_obj))


def get_mol_from_mols_pkl(target_nsc):
    if not isinstance(mols_obj, dict):
        return None
    
    target_nsc = int(target_nsc)
    
    # Direct / key-based check
    for key, mol in mols_obj.items():
        key_nsc = extract_nsc_from_text(key)
        if key_nsc == target_nsc:
            return mol
    
    # Property-based check
    for key, mol in mols_obj.items():
        possible_values = [key]
        
        if hasattr(mol, "GetPropNames"):
            try:
                possible_values.append(mol.GetProp("_Name"))
            except Exception:
                pass
            
            try:
                for prop in mol.GetPropNames():
                    possible_values.append(mol.GetProp(prop))
            except Exception:
                pass
        
        extracted = [extract_nsc_from_text(v) for v in possible_values]
        extracted = [v for v in extracted if v is not None]
        
        if target_nsc in extracted:
            return mol
    
    return None


def get_mol_from_sdf(target_nsc):
    target_nsc = int(target_nsc)
    
    supplier = Chem.SDMolSupplier(
        str(source_sdf_file),
        sanitize=False,
        removeHs=False
    )
    
    for mol in supplier:
        if mol is None:
            continue
        
        possible_values = []
        
        try:
            possible_values.append(mol.GetProp("_Name"))
        except Exception:
            pass
        
        try:
            for prop in mol.GetPropNames():
                possible_values.append(mol.GetProp(prop))
        except Exception:
            pass
        
        extracted = [extract_nsc_from_text(v) for v in possible_values]
        extracted = [v for v in extracted if v is not None]
        
        if target_nsc in extracted:
            return mol
    
    return None


# ------------------------------------------------------------
# 6. Recover missing feature rows
# ------------------------------------------------------------

recovered_rows = []
recovery_log = []

# ---- Recover NSC 119875 from local molecule files ----

if 119875 in missing_nscs:
    mol_119875 = get_mol_from_mols_pkl(119875)
    recovery_source = "drug_mols.pkl"
    
    if mol_119875 is None:
        mol_119875 = get_mol_from_sdf(119875)
        recovery_source = "ComboCompoundSet.sdf"
    
    if mol_119875 is None:
        recovery_log.append({
            "NSC": 119875,
            "status": "failed",
            "source": "",
            "note": "Molecule not found in local mols or SDF"
        })
    else:
        try:
            vector_119875 = generate_263_features_from_mol(mol_119875)
            recovered_rows.append(vector_to_feature_row(119875, vector_119875))
            
            recovery_log.append({
                "NSC": 119875,
                "status": "recovered",
                "source": recovery_source,
                "note": "Generated 263 features after fixing RDKit ring info"
            })
        except Exception as e:
            recovery_log.append({
                "NSC": 119875,
                "status": "failed",
                "source": recovery_source,
                "note": str(e)
            })


# ---- Recover NSC 753082 as Vemurafenib ----
# Try to copy NSC 761431 if available; otherwise generate from Vemurafenib SMILES.

if 753082 in missing_nscs:
    copied_from_761431 = False
    
    if (
        not source_features.empty
        and "NSC" in source_features.columns
        and 761431 in set(source_features["NSC"].unique())
        and all(col in source_features.columns for col in feature_cols)
    ):
        source_row = source_features[source_features["NSC"] == 761431].iloc[0].copy()
        source_row["NSC"] = 753082
        
        recovered_rows.append(source_row[["NSC"] + feature_cols].to_dict())
        copied_from_761431 = True
        
        recovery_log.append({
            "NSC": 753082,
            "status": "recovered",
            "source": "copied_from_NSC_761431",
            "note": "Copied features because NSC 761431 is also Vemurafenib"
        })
    
    if not copied_from_761431:
        vemurafenib_smiles = "CCCS(=O)(=O)Nc1ccc(c(c1F)C(=O)c1c[nH]c2c1cc(cn2)c1ccc(cc1)Cl)F"
        
        mol_753082 = Chem.MolFromSmiles(vemurafenib_smiles)
        
        if mol_753082 is None:
            recovery_log.append({
                "NSC": 753082,
                "status": "failed",
                "source": "Vemurafenib SMILES",
                "note": "RDKit could not parse Vemurafenib SMILES"
            })
        else:
            try:
                vector_753082 = generate_263_features_from_mol(mol_753082)
                recovered_rows.append(vector_to_feature_row(753082, vector_753082))
                
                recovery_log.append({
                    "NSC": 753082,
                    "status": "recovered",
                    "source": "Vemurafenib canonical SMILES",
                    "note": "Generated 263 features from Vemurafenib SMILES"
                })
            except Exception as e:
                recovery_log.append({
                    "NSC": 753082,
                    "status": "failed",
                    "source": "Vemurafenib SMILES",
                    "note": str(e)
                })


recovery_df = pd.DataFrame(recovery_log)

print("\n============================================================")
print("Recovery log")
print("============================================================")
display(recovery_df)


# ------------------------------------------------------------
# 7. Update feature table
# ------------------------------------------------------------

if len(recovered_rows) == 0:
    raise ValueError("No features were recovered. Cannot update feature table.")

recovered_features = pd.DataFrame(recovered_rows)

recovered_features["NSC"] = pd.to_numeric(
    recovered_features["NSC"],
    errors="coerce"
).astype(int)

for col in feature_cols:
    recovered_features[col] = pd.to_numeric(recovered_features[col], errors="coerce")

recovered_features = recovered_features[["NSC"] + feature_cols].copy()

updated_features = pd.concat(
    [features[["NSC"] + feature_cols], recovered_features],
    axis=0,
    ignore_index=True
)

updated_features = updated_features.drop_duplicates(subset=["NSC"], keep="last")
updated_features = updated_features.sort_values("NSC").reset_index(drop=True)

print("\nRecovered feature rows:")
display(recovered_features)


# ------------------------------------------------------------
# 8. Final coverage check
# ------------------------------------------------------------

updated_available_nscs = set(updated_features["NSC"].unique())
still_missing = sorted(needed_nscs - updated_available_nscs)

coverage_after = 100 * (len(needed_nscs) - len(still_missing)) / len(needed_nscs)

print("\n============================================================")
print("Coverage after Step 2C")
print("============================================================")

print("Drugs needed by FG-only ComboScore:", len(needed_nscs))
print("Drugs available after recovery:", len(updated_available_nscs))
print("Still missing:", still_missing)
print(f"Coverage after recovery: {coverage_after:.2f}%")

coverage_after_df = pd.DataFrame({
    "NSC": sorted(needed_nscs),
    "has_features_after_step2c": [
        nsc in updated_available_nscs for nsc in sorted(needed_nscs)
    ]
})

display(coverage_after_df["has_features_after_step2c"].value_counts())


# ------------------------------------------------------------
# 9. Save updated feature files
# ------------------------------------------------------------

updated_features.to_csv(current_features_file, index=False)
updated_features.to_pickle(DATA_DIR / "drug_features.pkl")

with open(feature_cols_json, "w") as f:
    json.dump(feature_cols, f, indent=2)

recovery_df.to_csv(recovery_report_file, index=False)
coverage_after_df.to_csv(coverage_after_file, index=False)

print("\n✅ Updated drug features saved:")
print(current_features_file)

print("\n✅ Updated drug features PKL saved:")
print(DATA_DIR / "drug_features.pkl")

print("\n✅ Recovery report saved:")
print(recovery_report_file)

print("\n✅ Coverage report saved:")
print(coverage_after_file)

print("\nFinal updated feature table shape:", updated_features.shape)

if len(still_missing) == 0:
    print("\n✅ STEP 2C COMPLETE — 100% feature coverage achieved")
else:
    print("\n⚠️ STEP 2C COMPLETE — still missing some drugs")

ComboScore file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\comboscore.csv
Current clean features: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv
Source features: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\drug_features.pkl
Source mols: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\drug_mols.pkl
Source SDF: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v2\data\ComboCompoundSet.sdf

Before recovery:
Drugs needed: 100
Drugs available: 98
Missing drugs: [119875, 753082]

Feature columns: 263

Source feature table shape: (103, 264)
drug_mols.pkl type: <class 'dict'>

Recovery log


[12:28:11] Explicit valence for atom # 0 Cl, 2, is greater than permitted


,NSC,status,source,note
0,119875,recovered,drug_mols.pkl,Generated 263 features after fixing RDKit ring...
1,753082,recovered,copied_from_NSC_761431,Copied features because NSC 761431 is also Vem...



Recovered feature rows:


,NSC,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_253,feat_254,feat_255,feat_256,feat_257,feat_258,feat_259,feat_260,feat_261,feat_262
0,119875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,298.030,-7.1757,52.040000,2.0,0.0,0.0,2.0
1,753082,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,489.931,5.5442,91.919998,4.0,0.0,2.0,4.0



Coverage after Step 2C
Drugs needed by FG-only ComboScore: 100
Drugs available after recovery: 100
Still missing: []
Coverage after recovery: 100.00%


has_features_after_step2c
True    100
Name: count, dtype: int64


✅ Updated drug features saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv

✅ Updated drug features PKL saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.pkl

✅ Recovery report saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step2c_feature_recovery_report.csv

✅ Coverage report saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step2c_coverage_after_recovery.csv

Final updated feature table shape: (100, 264)

✅ STEP 2C COMPLETE — 100% feature coverage achieved


In [10]:
# ============================================================
# STEP 3: Build model-ready matrix
# ============================================================

import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

comboscore_file = DATA_DIR / "comboscore.csv"
drug_features_file = DATA_DIR / "drug_features.csv"
feature_cols_json = DATA_DIR / "feature_columns.json"

model_matrix_file = DATA_DIR / "model_matrix.csv"
x_raw_file = DATA_DIR / "X_raw.npy"
y_raw_file = DATA_DIR / "y_raw.npy"
cell_lines_file = DATA_DIR / "cell_lines_raw.npy"
model_feature_cols_file = DATA_DIR / "model_feature_columns.json"

step3_report_file = RESULTS_DIR / "step3_model_matrix_report.csv"

print("ComboScore file:", comboscore_file)
print("Drug features file:", drug_features_file)
print("Output model matrix:", model_matrix_file)


# ------------------------------------------------------------
# 1. Load corrected ComboScore and recovered drug features
# ------------------------------------------------------------

comboscore = pd.read_csv(comboscore_file)
drug_features = pd.read_csv(drug_features_file)

print("\n✅ Files loaded")
print("ComboScore shape:", comboscore.shape)
print("Drug features shape:", drug_features.shape)


# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_combo_cols = ["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]

missing_combo_cols = [
    col for col in required_combo_cols
    if col not in comboscore.columns
]

if missing_combo_cols:
    raise ValueError(f"comboscore.csv missing columns: {missing_combo_cols}")

if "NSC" not in drug_features.columns:
    raise ValueError("drug_features.csv must contain NSC column.")

if feature_cols_json.exists():
    with open(feature_cols_json, "r") as f:
        drug_feature_cols = json.load(f)
else:
    drug_feature_cols = [c for c in drug_features.columns if c != "NSC"]

missing_feature_cols = [
    col for col in drug_feature_cols
    if col not in drug_features.columns
]

if missing_feature_cols:
    raise ValueError(f"drug_features.csv missing feature columns: {missing_feature_cols[:10]}")

print("\n✅ Required columns validated")
print("Drug feature columns:", len(drug_feature_cols))


# ------------------------------------------------------------
# 3. Clean datatypes
# ------------------------------------------------------------

comboscore["NSC1"] = pd.to_numeric(comboscore["NSC1"], errors="coerce")
comboscore["NSC2"] = pd.to_numeric(comboscore["NSC2"], errors="coerce")
comboscore["COMBOSCORE"] = pd.to_numeric(comboscore["COMBOSCORE"], errors="coerce")

before_combo_drop = len(comboscore)

comboscore = comboscore.dropna(
    subset=["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]
).copy()

after_combo_drop = len(comboscore)

comboscore["NSC1"] = comboscore["NSC1"].astype(int)
comboscore["NSC2"] = comboscore["NSC2"].astype(int)
comboscore["CELLNAME"] = comboscore["CELLNAME"].astype(str).str.strip()
comboscore["COMBOSCORE"] = comboscore["COMBOSCORE"].astype(np.float32)

drug_features["NSC"] = pd.to_numeric(drug_features["NSC"], errors="coerce")
drug_features = drug_features.dropna(subset=["NSC"]).copy()
drug_features["NSC"] = drug_features["NSC"].astype(int)

for col in drug_feature_cols:
    drug_features[col] = pd.to_numeric(drug_features[col], errors="coerce").astype(np.float32)

print("\n✅ Datatypes cleaned")
print("ComboScore rows dropped due to missing values:", before_combo_drop - after_combo_drop)


# ------------------------------------------------------------
# 4. Check duplicate drugs in feature table
# ------------------------------------------------------------

duplicate_drugs = drug_features["NSC"].duplicated().sum()

if duplicate_drugs > 0:
    print("\n⚠️ Duplicate NSC rows found in drug_features:", duplicate_drugs)
    print("Averaging duplicate feature rows by NSC...")
    drug_features = (
        drug_features
        .groupby("NSC", as_index=False)[drug_feature_cols]
        .mean()
    )
else:
    print("\n✅ No duplicate NSC rows in drug_features")


# ------------------------------------------------------------
# 5. Check 100% feature coverage before merging
# ------------------------------------------------------------

needed_nscs = set(comboscore["NSC1"]).union(set(comboscore["NSC2"]))
available_nscs = set(drug_features["NSC"])

missing_nscs = sorted(needed_nscs - available_nscs)

print("\n============================================================")
print("Feature coverage before merge")
print("============================================================")
print("Unique drugs needed:", len(needed_nscs))
print("Unique drugs available:", len(available_nscs))
print("Missing drugs:", missing_nscs)

if len(missing_nscs) > 0:
    raise ValueError(
        f"Cannot build model matrix. Missing features for NSCs: {missing_nscs}"
    )

print("✅ 100% feature coverage confirmed")


# ------------------------------------------------------------
# 6. Create D1 and D2 feature tables
# ------------------------------------------------------------

d1_features = drug_features.rename(
    columns={col: f"D1_{col}" for col in drug_feature_cols}
)

d2_features = drug_features.rename(
    columns={col: f"D2_{col}" for col in drug_feature_cols}
)

d1_feature_cols = [f"D1_{col}" for col in drug_feature_cols]
d2_feature_cols = [f"D2_{col}" for col in drug_feature_cols]

model_feature_cols = d1_feature_cols + d2_feature_cols

print("\nD1 feature columns:", len(d1_feature_cols))
print("D2 feature columns:", len(d2_feature_cols))
print("Total model input features:", len(model_feature_cols))


# ------------------------------------------------------------
# 7. Merge NSC1 features and NSC2 features
# ------------------------------------------------------------

row_count_before_merge = len(comboscore)

model_matrix = comboscore.merge(
    d1_features,
    left_on="NSC1",
    right_on="NSC",
    how="left"
)

model_matrix = model_matrix.drop(columns=["NSC"])

model_matrix = model_matrix.merge(
    d2_features,
    left_on="NSC2",
    right_on="NSC",
    how="left"
)

model_matrix = model_matrix.drop(columns=["NSC"])

row_count_after_merge = len(model_matrix)

print("\n✅ Drug features merged")
print("Rows before merge:", row_count_before_merge)
print("Rows after merge :", row_count_after_merge)

if row_count_before_merge != row_count_after_merge:
    raise ValueError("Row count changed after merge. Something is wrong.")


# ------------------------------------------------------------
# 8. Final missing-value check
# ------------------------------------------------------------

missing_feature_values = model_matrix[model_feature_cols].isna().sum().sum()
missing_target_values = model_matrix["COMBOSCORE"].isna().sum()

print("\nMissing feature values:", missing_feature_values)
print("Missing target values :", missing_target_values)

if missing_feature_values > 0:
    bad_rows = model_matrix[model_matrix[model_feature_cols].isna().any(axis=1)]
    raise ValueError(
        f"Model matrix still has missing feature values. Bad rows: {len(bad_rows)}"
    )

if missing_target_values > 0:
    raise ValueError("Model matrix has missing COMBOSCORE values.")

print("✅ No missing values in final model matrix")


# ------------------------------------------------------------
# 9. Reorder columns cleanly
# ------------------------------------------------------------

final_cols = ["NSC1", "NSC2", "CELLNAME"] + model_feature_cols + ["COMBOSCORE"]

model_matrix = model_matrix[final_cols].copy()

print("\n============================================================")
print("Final model matrix summary")
print("============================================================")
print("Final shape:", model_matrix.shape)
print("Expected input feature count:", len(model_feature_cols))
print("Unique cell lines:", model_matrix["CELLNAME"].nunique())
print("Unique drugs:", len(needed_nscs))
print("Unique drug pairs:", model_matrix[["NSC1", "NSC2"]].drop_duplicates().shape[0])

print("\nComboScore statistics:")
print(model_matrix["COMBOSCORE"].describe())

print("\nRows per cell line:")
print(model_matrix["CELLNAME"].value_counts().sort_index())


# ------------------------------------------------------------
# 10. Create NumPy arrays
# ------------------------------------------------------------

X_raw = model_matrix[model_feature_cols].to_numpy(dtype=np.float32)
y_raw = model_matrix["COMBOSCORE"].to_numpy(dtype=np.float32)
cell_lines_raw = model_matrix["CELLNAME"].to_numpy()

print("\n✅ NumPy arrays created")
print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)
print("cell_lines_raw shape:", cell_lines_raw.shape)


# ------------------------------------------------------------
# 11. Save all Step 3 outputs into v3/data
# ------------------------------------------------------------

model_matrix.to_csv(model_matrix_file, index=False)

np.save(x_raw_file, X_raw)
np.save(y_raw_file, y_raw)
np.save(cell_lines_file, cell_lines_raw)

with open(model_feature_cols_file, "w") as f:
    json.dump(model_feature_cols, f, indent=2)

step3_report = pd.DataFrame([{
    "model_matrix_rows": model_matrix.shape[0],
    "model_matrix_columns": model_matrix.shape[1],
    "input_feature_count": len(model_feature_cols),
    "drug_feature_count_per_drug": len(drug_feature_cols),
    "unique_cell_lines": model_matrix["CELLNAME"].nunique(),
    "unique_drugs": len(needed_nscs),
    "unique_drug_pairs": model_matrix[["NSC1", "NSC2"]].drop_duplicates().shape[0],
    "missing_feature_values": int(missing_feature_values),
    "missing_target_values": int(missing_target_values),
    "comboscore_min": float(model_matrix["COMBOSCORE"].min()),
    "comboscore_max": float(model_matrix["COMBOSCORE"].max()),
    "comboscore_mean": float(model_matrix["COMBOSCORE"].mean()),
    "comboscore_std": float(model_matrix["COMBOSCORE"].std())
}])

step3_report.to_csv(step3_report_file, index=False)

print("\n✅ Step 3 files saved")
print("Model matrix CSV:", model_matrix_file)
print("X_raw.npy:", x_raw_file)
print("y_raw.npy:", y_raw_file)
print("cell_lines_raw.npy:", cell_lines_file)
print("Model feature columns:", model_feature_cols_file)
print("Step 3 report:", step3_report_file)

print("\nPreview:")
display(model_matrix.head())

print("\n✅ STEP 3 COMPLETE")

ComboScore file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\comboscore.csv
Drug features file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv
Output model matrix: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_matrix.csv

✅ Files loaded
ComboScore shape: (145212, 4)
Drug features shape: (100, 264)

✅ Required columns validated
Drug feature columns: 263

✅ Datatypes cleaned
ComboScore rows dropped due to missing values: 0

✅ No duplicate NSC rows in drug_features

Feature coverage before merge
Unique drugs needed: 100
Unique drugs available: 100
Missing drugs: []
✅ 100% feature coverage confirmed

D1 feature columns: 263
D2 feature columns: 263
Total model input features: 526

✅ Drug features merged
Rows before merge: 145212
Rows after merge : 145212

Missing feature values: 0
Missing target values : 0
✅ No missing values in final model matrix

Final model matrix summary
Final shape: (145212, 530)
Expected input feature count: 526
Unique cel

,NSC1,NSC2,CELLNAME,D1_feat_0,D1_feat_1,D1_feat_2,D1_feat_3,D1_feat_4,D1_feat_5,D1_feat_6,...,D2_feat_254,D2_feat_255,D2_feat_256,D2_feat_257,D2_feat_258,D2_feat_259,D2_feat_260,D2_feat_261,D2_feat_262,COMBOSCORE
0,740,752,786-0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,167.197006,0.59769,83.379997,2.0,0.0,3.0,4.0,-80.0
1,740,3088,786-0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,304.217010,3.37790,40.540001,1.0,0.0,1.0,2.0,-44.0
2,740,8806,786-0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,341.665985,2.34680,66.559998,1.0,0.0,2.0,3.0,-64.0
3,740,13875,786-0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,210.285004,0.06960,48.389999,1.0,0.0,0.0,6.0,-14.0
4,740,19893,786-0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,130.078003,-0.79770,65.720001,1.0,0.0,2.0,2.0,-54.0



✅ STEP 3 COMPLETE


In [12]:
%pip install xgboost catboost lightgbm scipy

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---

In [13]:
# ============================================================
# STEP 4: One-cell-line training with 4 models
# Models: Random Forest, XGBoost, CatBoost, LightGBM
# ============================================================

import json
import time
import re
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ------------------------------------------------------------
# 0. Check external ML libraries
# ------------------------------------------------------------

missing_packages = []

try:
    from xgboost import XGBRegressor
except Exception:
    missing_packages.append("xgboost")

try:
    from catboost import CatBoostRegressor
except Exception:
    missing_packages.append("catboost")

try:
    from lightgbm import LGBMRegressor
except Exception:
    missing_packages.append("lightgbm")

try:
    from scipy.stats import pearsonr, spearmanr
except Exception:
    missing_packages.append("scipy")

if missing_packages:
    print("❌ Missing packages:", missing_packages)
    print("\nRun this in a new notebook cell first:")
    print("%pip install xgboost catboost lightgbm scipy")
    raise ImportError("Install missing packages before running Step 4.")


# ------------------------------------------------------------
# 1. Settings
# ------------------------------------------------------------

SELECTED_CELL_LINE = "786-0"
RANDOM_STATE = 42
TEST_SIZE = 0.10

model_matrix_file = DATA_DIR / "model_matrix.csv"
model_feature_cols_file = DATA_DIR / "model_feature_columns.json"

metrics_file = RESULTS_DIR / "step4_one_cellline_model_comparison.csv"
all_predictions_file = RESULTS_DIR / "step4_one_cellline_all_predictions.csv"

print("Selected cell line:", SELECTED_CELL_LINE)
print("Model matrix file :", model_matrix_file)
print("Feature cols file :", model_feature_cols_file)


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def safe_filename(text):
    """
    Converts cell-line names into safe filenames.
    Example:
        786-0 -> 786_0
        MDA-MB-231/ATCC -> MDA_MB_231_ATCC
    """
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    text = text.strip("_")
    return text


def compute_metrics(y_true, y_pred):
    """
    Computes paper-style and sklearn-style regression metrics.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        rp = np.nan
        rs = np.nan
    else:
        rp = pearsonr(y_true, y_pred)[0]
        rs = spearmanr(y_true, y_pred)[0]

    return {
        "r2_score": r2,
        "rmse": rmse,
        "mae": mae,
        "pearson_rp": rp,
        "spearman_rs": rs
    }


def augment_reverse_drug_order(df, d1_cols, d2_cols):
    """
    Training-only augmentation.

    Original:
        NSC1 + NSC2
        D1 features + D2 features

    Reversed:
        NSC2 + NSC1
        D2 features + D1 features

    COMBOSCORE stays same.
    """
    original = df.copy()
    reversed_df = df.copy()

    # Swap NSC IDs
    reversed_df[["NSC1", "NSC2"]] = reversed_df[["NSC2", "NSC1"]].to_numpy()

    # Swap D1 and D2 feature values
    temp_d1 = reversed_df[d1_cols].copy()
    reversed_df[d1_cols] = reversed_df[d2_cols].to_numpy()
    reversed_df[d2_cols] = temp_d1.to_numpy()

    augmented = pd.concat([original, reversed_df], axis=0, ignore_index=True)

    return augmented


# ------------------------------------------------------------
# 3. Load model matrix and feature columns
# ------------------------------------------------------------

model_matrix = pd.read_csv(model_matrix_file)

with open(model_feature_cols_file, "r") as f:
    model_feature_cols = json.load(f)

print("\n✅ Files loaded")
print("Model matrix shape:", model_matrix.shape)
print("Input feature count:", len(model_feature_cols))

required_cols = ["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]

for col in required_cols:
    if col not in model_matrix.columns:
        raise ValueError(f"Missing required column in model_matrix.csv: {col}")

missing_model_cols = [c for c in model_feature_cols if c not in model_matrix.columns]

if missing_model_cols:
    raise ValueError(f"Missing model feature columns: {missing_model_cols[:10]}")


# ------------------------------------------------------------
# 4. Select one cell line
# ------------------------------------------------------------

available_cell_lines = sorted(model_matrix["CELLNAME"].unique())

if SELECTED_CELL_LINE not in available_cell_lines:
    print("Available cell lines:")
    print(available_cell_lines)
    raise ValueError(f"Cell line not found: {SELECTED_CELL_LINE}")

cell_df = model_matrix[model_matrix["CELLNAME"] == SELECTED_CELL_LINE].copy()
cell_df = cell_df.reset_index(drop=True)

print("\n============================================================")
print("Selected cell-line data")
print("============================================================")
print("Cell line:", SELECTED_CELL_LINE)
print("Rows:", len(cell_df))
print("Unique drugs:", len(set(cell_df["NSC1"]).union(set(cell_df["NSC2"]))))
print("Unique drug pairs:", cell_df[["NSC1", "NSC2"]].drop_duplicates().shape[0])

print("\nCOMBOSCORE stats:")
print(cell_df["COMBOSCORE"].describe())


# ------------------------------------------------------------
# 5. Prepare D1/D2 feature column groups
# ------------------------------------------------------------

d1_cols = [c for c in model_feature_cols if c.startswith("D1_")]
d2_cols = ["D2_" + c.replace("D1_", "", 1) for c in d1_cols]

if len(d1_cols) != 263:
    raise ValueError(f"Expected 263 D1 features, found {len(d1_cols)}")

if len(d2_cols) != 263:
    raise ValueError(f"Expected 263 D2 features, found {len(d2_cols)}")

for col in d2_cols:
    if col not in model_feature_cols:
        raise ValueError(f"Missing matching D2 column for augmentation: {col}")

print("\n✅ Feature groups ready")
print("D1 columns:", len(d1_cols))
print("D2 columns:", len(d2_cols))


# ------------------------------------------------------------
# 6. 90/10 train-test split
# ------------------------------------------------------------

train_df, test_df = train_test_split(
    cell_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\n============================================================")
print("Train-test split")
print("============================================================")
print("Train rows before augmentation:", len(train_df))
print("Test rows:", len(test_df))


# ------------------------------------------------------------
# 7. Augment only training data
# ------------------------------------------------------------

train_aug_df = augment_reverse_drug_order(train_df, d1_cols, d2_cols)

print("\n============================================================")
print("Training augmentation")
print("============================================================")
print("Train rows before augmentation:", len(train_df))
print("Train rows after augmentation :", len(train_aug_df))
print("Test rows remain untouched    :", len(test_df))


# ------------------------------------------------------------
# 8. Create X/y arrays
# ------------------------------------------------------------

X_train = train_aug_df[model_feature_cols].to_numpy(dtype=np.float32)
y_train = train_aug_df["COMBOSCORE"].to_numpy(dtype=np.float32)

X_test = test_df[model_feature_cols].to_numpy(dtype=np.float32)
y_test = test_df["COMBOSCORE"].to_numpy(dtype=np.float32)

print("\n✅ X/y arrays ready")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ------------------------------------------------------------
# 9. Define 4 models
# ------------------------------------------------------------

n_features = X_train.shape[1]
rf_max_features = max(1, n_features // 3)

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=250,
        max_features=rf_max_features,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist"
    ),

    "CatBoost": CatBoostRegressor(
        iterations=700,
        depth=6,
        learning_rate=0.03,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=700,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )
}

print("\n✅ Models defined")
print("Models:", list(models.keys()))


# ------------------------------------------------------------
# 10. Train, predict, evaluate, save
# ------------------------------------------------------------

safe_cell = safe_filename(SELECTED_CELL_LINE)

metrics_rows = []
all_predictions = []

for model_name, model in models.items():
    print("\n============================================================")
    print(f"Training model: {model_name}")
    print("============================================================")

    start_time = time.time()

    model.fit(X_train, y_train)

    train_time_sec = time.time() - start_time

    y_pred = model.predict(X_test)

    metrics = compute_metrics(y_test, y_pred)

    model_file = MODELS_DIR / f"step4_{model_name.lower()}_{safe_cell}.pkl"
    joblib.dump(model, model_file)

    pred_df = test_df[["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]].copy()
    pred_df = pred_df.rename(columns={"COMBOSCORE": "actual_comboscore"})
    pred_df["predicted_comboscore"] = y_pred
    pred_df["prediction_error"] = pred_df["actual_comboscore"] - pred_df["predicted_comboscore"]
    pred_df["model"] = model_name

    pred_file = RESULTS_DIR / f"step4_predictions_{model_name.lower()}_{safe_cell}.csv"
    pred_df.to_csv(pred_file, index=False)

    all_predictions.append(pred_df)

    metrics_row = {
        "cell_line": SELECTED_CELL_LINE,
        "model": model_name,
        "train_rows_before_augmentation": len(train_df),
        "train_rows_after_augmentation": len(train_aug_df),
        "test_rows": len(test_df),
        "input_features": len(model_feature_cols),
        "r2_score": metrics["r2_score"],
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "pearson_rp": metrics["pearson_rp"],
        "spearman_rs": metrics["spearman_rs"],
        "train_time_sec": train_time_sec,
        "model_path": str(model_file),
        "prediction_file": str(pred_file)
    }

    metrics_rows.append(metrics_row)

    print(f"✅ {model_name} complete")
    print("R² score   :", round(metrics["r2_score"], 4))
    print("RMSE       :", round(metrics["rmse"], 4))
    print("MAE        :", round(metrics["mae"], 4))
    print("Pearson Rp :", round(metrics["pearson_rp"], 4))
    print("Spearman Rs:", round(metrics["spearman_rs"], 4))
    print("Train time :", round(train_time_sec, 2), "sec")
    print("Model saved:", model_file)
    print("Predictions:", pred_file)


# ------------------------------------------------------------
# 11. Save comparison files
# ------------------------------------------------------------

metrics_df = pd.DataFrame(metrics_rows)

metrics_df = metrics_df.sort_values(
    by="r2_score",
    ascending=False
).reset_index(drop=True)

metrics_df.to_csv(metrics_file, index=False)

all_predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True)
all_predictions_df.to_csv(all_predictions_file, index=False)

print("\n============================================================")
print("STEP 4 MODEL COMPARISON")
print("============================================================")

display(metrics_df)

print("\n✅ Step 4 metrics saved:")
print(metrics_file)

print("\n✅ All predictions saved:")
print(all_predictions_file)

print("\nBest model by R²:")
best_row = metrics_df.iloc[0]
print("Model:", best_row["model"])
print("R²   :", round(best_row["r2_score"], 4))
print("Rp   :", round(best_row["pearson_rp"], 4))

print("\n✅ STEP 4 COMPLETE")

Selected cell line: 786-0
Model matrix file : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_matrix.csv
Feature cols file : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_feature_columns.json

✅ Files loaded
Model matrix shape: (145212, 530)
Input feature count: 526

Selected cell-line data
Cell line: 786-0
Rows: 2439
Unique drugs: 100
Unique drug pairs: 2439

COMBOSCORE stats:
count    2439.000000
mean       -6.020500
std        41.731283
min      -349.000000
25%       -29.500000
50%        -7.000000
75%        17.000000
max       202.000000
Name: COMBOSCORE, dtype: float64

✅ Feature groups ready
D1 columns: 263
D2 columns: 263

Train-test split
Train rows before augmentation: 2195
Test rows: 244

Training augmentation
Train rows before augmentation: 2195
Train rows after augmentation : 4390
Test rows remain untouched    : 244

✅ X/y arrays ready
X_train: (4390, 526)
y_train: (4390,)
X_test : (244, 526)
y_test : (244,)

✅ Models defined
Models: ['RandomForest'

,cell_line,model,train_rows_before_augmentation,train_rows_after_augmentation,test_rows,input_features,r2_score,rmse,mae,pearson_rp,spearman_rs,train_time_sec,model_path,prediction_file
0,786-0,CatBoost,2195,4390,244,526,0.341898,30.885618,24.368193,0.585791,0.587811,6.830363,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
1,786-0,RandomForest,2195,4390,244,526,0.334882,31.049815,24.507180,0.579649,0.586264,15.983803,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
2,786-0,XGBoost,2195,4390,244,526,0.332741,31.099740,24.307348,0.580476,0.603168,5.200113,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
3,786-0,LightGBM,2195,4390,244,526,0.307892,31.673525,24.320793,0.571636,0.580073,4.742628,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...



✅ Step 4 metrics saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step4_one_cellline_model_comparison.csv

✅ All predictions saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step4_one_cellline_all_predictions.csv

Best model by R²:
Model: CatBoost
R²   : 0.3419
Rp   : 0.5858

✅ STEP 4 COMPLETE


In [14]:
# ============================================================
# STEP 5: Train all 60 cell-line models with 4 models
# Models: RandomForest, XGBoost, CatBoost, LightGBM
# ============================================================

import json
import time
import re
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr, spearmanr


# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

RANDOM_STATE = 42
TEST_SIZE = 0.10

SAVE_MODELS = True
SAVE_PREDICTIONS = True

model_matrix_file = DATA_DIR / "model_matrix.csv"
model_feature_cols_file = DATA_DIR / "model_feature_columns.json"

all_metrics_file = RESULTS_DIR / "step5_all_cellline_model_comparison.csv"
best_per_cellline_file = RESULTS_DIR / "step5_best_model_per_cellline.csv"
average_model_perf_file = RESULTS_DIR / "step5_average_model_performance.csv"
all_predictions_file = RESULTS_DIR / "step5_all_cellline_all_model_predictions.csv"

print("Model matrix file:", model_matrix_file)
print("Feature cols file:", model_feature_cols_file)
print("Save models:", SAVE_MODELS)
print("Save predictions:", SAVE_PREDICTIONS)


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def safe_filename(text):
    """
    Makes safe filenames from cell-line names.
    Example:
        786-0 -> 786_0
        MDA-MB-231/ATCC -> MDA_MB_231_ATCC
    """
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    text = text.strip("_")
    return text


def compute_metrics(y_true, y_pred):
    """
    Calculates regression metrics:
    R², RMSE, MAE, Pearson Rp, Spearman Rs.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        rp = np.nan
        rs = np.nan
    else:
        rp = pearsonr(y_true, y_pred)[0]
        rs = spearmanr(y_true, y_pred)[0]

    return {
        "r2_score": r2,
        "rmse": rmse,
        "mae": mae,
        "pearson_rp": rp,
        "spearman_rs": rs
    }


def augment_reverse_drug_order(df, d1_cols, d2_cols):
    """
    Training-only augmentation.

    Original:
        NSC1 + NSC2
        D1 features + D2 features

    Reversed:
        NSC2 + NSC1
        D2 features + D1 features

    COMBOSCORE remains same.
    """
    original = df.copy()
    reversed_df = df.copy()

    # Swap NSC IDs
    reversed_df[["NSC1", "NSC2"]] = reversed_df[["NSC2", "NSC1"]].to_numpy()

    # Swap D1 and D2 feature vectors
    temp_d1 = reversed_df[d1_cols].copy()
    reversed_df[d1_cols] = reversed_df[d2_cols].to_numpy()
    reversed_df[d2_cols] = temp_d1.to_numpy()

    augmented = pd.concat([original, reversed_df], axis=0, ignore_index=True)

    return augmented


def make_models(random_state, n_features):
    """
    Creates fresh unfitted models for each cell line.
    """
    rf_max_features = max(1, n_features // 3)

    models = {
        "RandomForest": RandomForestRegressor(
            n_estimators=250,
            max_features=rf_max_features,
            random_state=random_state,
            n_jobs=-1
        ),

        "XGBoost": XGBRegressor(
            n_estimators=700,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist"
        ),

        "CatBoost": CatBoostRegressor(
            iterations=700,
            depth=6,
            learning_rate=0.03,
            loss_function="RMSE",
            random_seed=random_state,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        ),

        "LightGBM": LGBMRegressor(
            n_estimators=700,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1
        )
    }

    return models


# ------------------------------------------------------------
# 2. Load model matrix and feature columns
# ------------------------------------------------------------

model_matrix = pd.read_csv(model_matrix_file)

with open(model_feature_cols_file, "r") as f:
    model_feature_cols = json.load(f)

print("\n✅ Files loaded")
print("Model matrix shape:", model_matrix.shape)
print("Input feature count:", len(model_feature_cols))

required_cols = ["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]

for col in required_cols:
    if col not in model_matrix.columns:
        raise ValueError(f"Missing required column: {col}")

missing_feature_cols = [c for c in model_feature_cols if c not in model_matrix.columns]

if missing_feature_cols:
    raise ValueError(f"Missing model feature columns: {missing_feature_cols[:10]}")


# ------------------------------------------------------------
# 3. Prepare D1/D2 columns for augmentation
# ------------------------------------------------------------

d1_cols = [c for c in model_feature_cols if c.startswith("D1_")]
d2_cols = ["D2_" + c.replace("D1_", "", 1) for c in d1_cols]

if len(d1_cols) != 263:
    raise ValueError(f"Expected 263 D1 columns, found {len(d1_cols)}")

if len(d2_cols) != 263:
    raise ValueError(f"Expected 263 D2 columns, found {len(d2_cols)}")

for col in d2_cols:
    if col not in model_feature_cols:
        raise ValueError(f"Missing matching D2 column: {col}")

print("\n✅ Feature groups ready")
print("D1 columns:", len(d1_cols))
print("D2 columns:", len(d2_cols))


# ------------------------------------------------------------
# 4. Get all cell lines
# ------------------------------------------------------------

cell_lines = sorted(model_matrix["CELLNAME"].unique())

print("\n============================================================")
print("Step 5 training setup")
print("============================================================")
print("Total cell lines:", len(cell_lines))
print("Models per cell line: 4")
print("Total models to train:", len(cell_lines) * 4)
print("Cell lines:")
print(cell_lines)


# ------------------------------------------------------------
# 5. Main training loop
# ------------------------------------------------------------

all_metrics_rows = []
all_prediction_dfs = []

overall_start_time = time.time()

for cell_index, cell_line in enumerate(cell_lines, start=1):

    safe_cell = safe_filename(cell_line)

    print("\n\n############################################################")
    print(f"Cell line {cell_index}/{len(cell_lines)}: {cell_line}")
    print("############################################################")

    # ------------------------------
    # Filter one cell line
    # ------------------------------

    cell_df = model_matrix[model_matrix["CELLNAME"] == cell_line].copy()
    cell_df = cell_df.reset_index(drop=True)

    print("Rows:", len(cell_df))
    print("Unique drugs:", len(set(cell_df["NSC1"]).union(set(cell_df["NSC2"]))))
    print("Unique drug pairs:", cell_df[["NSC1", "NSC2"]].drop_duplicates().shape[0])

    if len(cell_df) < 100:
        print("⚠️ Skipping because too few rows.")
        continue

    # ------------------------------
    # 90/10 train-test split
    # ------------------------------

    train_df, test_df = train_test_split(
        cell_df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    # ------------------------------
    # Augment training only
    # ------------------------------

    train_aug_df = augment_reverse_drug_order(train_df, d1_cols, d2_cols)

    print("Train rows before augmentation:", len(train_df))
    print("Train rows after augmentation :", len(train_aug_df))
    print("Test rows:", len(test_df))

    # ------------------------------
    # Create X/y
    # ------------------------------

    X_train = train_aug_df[model_feature_cols].to_numpy(dtype=np.float32)
    y_train = train_aug_df["COMBOSCORE"].to_numpy(dtype=np.float32)

    X_test = test_df[model_feature_cols].to_numpy(dtype=np.float32)
    y_test = test_df["COMBOSCORE"].to_numpy(dtype=np.float32)

    # ------------------------------
    # Fresh models for this cell line
    # ------------------------------

    models = make_models(
        random_state=RANDOM_STATE,
        n_features=X_train.shape[1]
    )

    # ------------------------------
    # Train 4 models
    # ------------------------------

    for model_name, model in models.items():

        print("\n------------------------------------------------------------")
        print(f"Training {model_name} for {cell_line}")
        print("------------------------------------------------------------")

        model_start_time = time.time()

        try:
            model.fit(X_train, y_train)

            train_time_sec = time.time() - model_start_time

            y_pred = model.predict(X_test)

            metrics = compute_metrics(y_test, y_pred)

            # Save model
            if SAVE_MODELS:
                model_file = MODELS_DIR / f"step5_{model_name.lower()}_{safe_cell}.pkl"
                joblib.dump(model, model_file)
            else:
                model_file = ""

            # Save predictions
            pred_df = test_df[["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]].copy()
            pred_df = pred_df.rename(columns={"COMBOSCORE": "actual_comboscore"})
            pred_df["predicted_comboscore"] = y_pred
            pred_df["prediction_error"] = (
                pred_df["actual_comboscore"] - pred_df["predicted_comboscore"]
            )
            pred_df["model"] = model_name

            if SAVE_PREDICTIONS:
                pred_file = RESULTS_DIR / f"step5_predictions_{model_name.lower()}_{safe_cell}.csv"
                pred_df.to_csv(pred_file, index=False)
            else:
                pred_file = ""

            all_prediction_dfs.append(pred_df)

            metrics_row = {
                "cell_line": cell_line,
                "model": model_name,
                "rows_total": len(cell_df),
                "train_rows_before_augmentation": len(train_df),
                "train_rows_after_augmentation": len(train_aug_df),
                "test_rows": len(test_df),
                "input_features": len(model_feature_cols),
                "r2_score": metrics["r2_score"],
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "pearson_rp": metrics["pearson_rp"],
                "spearman_rs": metrics["spearman_rs"],
                "train_time_sec": train_time_sec,
                "model_path": str(model_file),
                "prediction_file": str(pred_file),
                "status": "success",
                "error": ""
            }

            all_metrics_rows.append(metrics_row)

            print(f"✅ {model_name} done")
            print("R² score   :", round(metrics["r2_score"], 4))
            print("RMSE       :", round(metrics["rmse"], 4))
            print("MAE        :", round(metrics["mae"], 4))
            print("Pearson Rp :", round(metrics["pearson_rp"], 4))
            print("Spearman Rs:", round(metrics["spearman_rs"], 4))
            print("Train time :", round(train_time_sec, 2), "sec")

        except Exception as e:
            train_time_sec = time.time() - model_start_time

            print(f"❌ {model_name} failed for {cell_line}")
            print("Error:", str(e))

            metrics_row = {
                "cell_line": cell_line,
                "model": model_name,
                "rows_total": len(cell_df),
                "train_rows_before_augmentation": len(train_df),
                "train_rows_after_augmentation": len(train_aug_df),
                "test_rows": len(test_df),
                "input_features": len(model_feature_cols),
                "r2_score": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "pearson_rp": np.nan,
                "spearman_rs": np.nan,
                "train_time_sec": train_time_sec,
                "model_path": "",
                "prediction_file": "",
                "status": "failed",
                "error": str(e)
            }

            all_metrics_rows.append(metrics_row)

    # Save progress after every cell line
    progress_df = pd.DataFrame(all_metrics_rows)
    progress_df.to_csv(all_metrics_file, index=False)

    print("\n✅ Progress saved after cell line:", cell_line)


# ------------------------------------------------------------
# 6. Save final metrics
# ------------------------------------------------------------

total_time_sec = time.time() - overall_start_time

metrics_df = pd.DataFrame(all_metrics_rows)

metrics_df = metrics_df.sort_values(
    by=["cell_line", "r2_score"],
    ascending=[True, False]
).reset_index(drop=True)

metrics_df.to_csv(all_metrics_file, index=False)

print("\n\n============================================================")
print("STEP 5 COMPLETE: ALL MODEL METRICS")
print("============================================================")
print("Total training time:", round(total_time_sec / 60, 2), "minutes")
print("Metrics shape:", metrics_df.shape)

display(metrics_df.head(20))


# ------------------------------------------------------------
# 7. Best model per cell line
# ------------------------------------------------------------

successful_metrics = metrics_df[metrics_df["status"] == "success"].copy()

best_per_cellline = (
    successful_metrics
    .sort_values(["cell_line", "r2_score"], ascending=[True, False])
    .groupby("cell_line", as_index=False)
    .first()
)

best_per_cellline.to_csv(best_per_cellline_file, index=False)

print("\n============================================================")
print("Best model per cell line")
print("============================================================")
print("Shape:", best_per_cellline.shape)

display(best_per_cellline[[
    "cell_line",
    "model",
    "r2_score",
    "rmse",
    "mae",
    "pearson_rp",
    "spearman_rs"
]].head(20))


# ------------------------------------------------------------
# 8. Average performance per model
# ------------------------------------------------------------

average_model_perf = (
    successful_metrics
    .groupby("model", as_index=False)
    .agg(
        mean_r2_score=("r2_score", "mean"),
        median_r2_score=("r2_score", "median"),
        mean_rmse=("rmse", "mean"),
        mean_mae=("mae", "mean"),
        mean_pearson_rp=("pearson_rp", "mean"),
        median_pearson_rp=("pearson_rp", "median"),
        mean_spearman_rs=("spearman_rs", "mean"),
        total_train_time_sec=("train_time_sec", "sum"),
        successful_cellline_count=("cell_line", "nunique")
    )
    .sort_values("mean_r2_score", ascending=False)
    .reset_index(drop=True)
)

average_model_perf.to_csv(average_model_perf_file, index=False)

print("\n============================================================")
print("Average model performance across all cell lines")
print("============================================================")

display(average_model_perf)


# ------------------------------------------------------------
# 9. Save combined predictions
# ------------------------------------------------------------

if len(all_prediction_dfs) > 0:
    all_predictions_df = pd.concat(all_prediction_dfs, axis=0, ignore_index=True)
    all_predictions_df.to_csv(all_predictions_file, index=False)

    print("\n✅ Combined predictions saved:")
    print(all_predictions_file)
    print("All predictions shape:", all_predictions_df.shape)
else:
    print("\n⚠️ No predictions were saved.")


# ------------------------------------------------------------
# 10. Final saved files summary
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 5 SAVED FILES")
print("============================================================")
print("All model comparison:", all_metrics_file)
print("Best model per cell line:", best_per_cellline_file)
print("Average model performance:", average_model_perf_file)

if SAVE_PREDICTIONS:
    print("Combined predictions:", all_predictions_file)

print("\n✅ STEP 5 COMPLETE")

Model matrix file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_matrix.csv
Feature cols file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_feature_columns.json
Save models: True
Save predictions: True

✅ Files loaded
Model matrix shape: (145212, 530)
Input feature count: 526

✅ Feature groups ready
D1 columns: 263
D2 columns: 263

Step 5 training setup
Total cell lines: 60
Models per cell line: 4
Total models to train: 240
Cell lines:
['786-0', 'A498', 'A549/ATCC', 'ACHN', 'BT-549', 'CAKI-1', 'CCRF-CEM', 'COLO 205', 'DU-145', 'EKVX', 'HCC-2998', 'HCT-116', 'HCT-15', 'HL-60(TB)', 'HOP-62', 'HOP-92', 'HS 578T', 'HT29', 'IGROV1', 'K-562', 'KM12', 'LOX IMVI', 'M14', 'MALME-3M', 'MCF7', 'MDA-MB-231/ATCC', 'MDA-MB-435', 'MDA-MB-468', 'MOLT-4', 'NCI-H226', 'NCI-H23', 'NCI-H322M', 'NCI-H460', 'NCI-H522', 'NCI/ADR-RES', 'OVCAR-3', 'OVCAR-4', 'OVCAR-5', 'OVCAR-8', 'PC-3', 'RPMI-8226', 'RXF 393', 'SF-268', 'SF-295', 'SF-539', 'SK-MEL-2', 'SK-MEL-28', 'SK-MEL-5', 'SK-OV

,cell_line,model,rows_total,train_rows_before_augmentation,train_rows_after_augmentation,test_rows,input_features,r2_score,rmse,mae,pearson_rp,spearman_rs,train_time_sec,model_path,prediction_file,status,error
0,786-0,CatBoost,2439,2195,4390,244,526,0.341898,30.885618,24.368193,0.585791,0.587811,5.408880,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
1,786-0,RandomForest,2439,2195,4390,244,526,0.334882,31.049815,24.507180,0.579649,0.586264,12.179025,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
2,786-0,XGBoost,2439,2195,4390,244,526,0.332741,31.099740,24.307348,0.580476,0.603168,2.838003,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
3,786-0,LightGBM,2439,2195,4390,244,526,0.307892,31.673525,24.320793,0.571636,0.580073,2.904668,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
4,A498,CatBoost,2441,2196,4392,245,526,0.447089,33.008112,23.161057,0.673602,0.641828,5.162141,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
5,A498,XGBoost,2441,2196,4392,245,526,0.438346,33.268073,23.594448,0.662911,0.624746,2.656190,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
6,A498,RandomForest,2441,2196,4392,245,526,0.417386,33.883134,23.528179,0.647812,0.637782,10.796582,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
7,A498,LightGBM,2441,2196,4392,245,526,0.407777,34.161399,24.077546,0.638919,0.630590,2.990115,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
8,A549/ATCC,LightGBM,2439,2195,4390,244,526,0.395637,29.374134,21.848186,0.629480,0.584564,2.669145,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
9,A549/ATCC,RandomForest,2439,2195,4390,244,526,0.358398,30.265582,21.945911,0.602748,0.587779,11.165902,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,



Best model per cell line
Shape: (60, 17)


,cell_line,model,r2_score,rmse,mae,pearson_rp,spearman_rs
0,786-0,CatBoost,0.341898,30.885618,24.368193,0.585791,0.587811
1,A498,CatBoost,0.447089,33.008112,23.161057,0.673602,0.641828
2,A549/ATCC,LightGBM,0.395637,29.374134,21.848186,0.629480,0.584564
3,ACHN,RandomForest,0.371066,42.416528,31.872216,0.623041,0.564880
4,BT-549,RandomForest,0.336038,48.121570,33.925945,0.585430,0.542594
5,CAKI-1,LightGBM,0.378873,43.512602,30.896801,0.622691,0.597871
6,CCRF-CEM,RandomForest,0.386014,57.319395,42.582848,0.627295,0.513332
7,COLO 205,CatBoost,0.341847,47.043135,35.993181,0.588019,0.532671
8,DU-145,XGBoost,0.434345,44.379482,27.544412,0.662433,0.616230
9,EKVX,XGBoost,0.513032,37.043191,26.724034,0.739068,0.649060



Average model performance across all cell lines


,model,mean_r2_score,median_r2_score,mean_rmse,mean_mae,mean_pearson_rp,median_pearson_rp,mean_spearman_rs,total_train_time_sec,successful_cellline_count
0,CatBoost,0.374818,0.354692,44.221958,31.615502,0.617905,0.616092,0.572928,299.560889,60
1,RandomForest,0.371234,0.366933,44.317994,31.315135,0.612695,0.614800,0.584765,660.304590,60
2,LightGBM,0.370570,0.372275,44.290690,31.475354,0.612983,0.618274,0.580449,177.991081,60
3,XGBoost,0.369660,0.355505,44.422604,31.646120,0.611927,0.602964,0.575614,153.826517,60



✅ Combined predictions saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_all_cellline_all_model_predictions.csv
All predictions shape: (58180, 7)

STEP 5 SAVED FILES
All model comparison: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_all_cellline_model_comparison.csv
Best model per cell line: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_best_model_per_cellline.csv
Average model performance: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_average_model_performance.csv
Combined predictions: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_all_cellline_all_model_predictions.csv

✅ STEP 5 COMPLETE


# Step 6A and 6B-E are not importent they reduced models performance 

In [15]:
# ============================================================
# STEP 6A: Diagnose Step 5 results
# ============================================================

import pandas as pd
import numpy as np

step5_metrics_file = RESULTS_DIR / "step5_all_cellline_model_comparison.csv"
step5_best_file = RESULTS_DIR / "step5_best_model_per_cellline.csv"
step5_avg_file = RESULTS_DIR / "step5_average_model_performance.csv"

diagnostics_file = RESULTS_DIR / "step6a_diagnostics_summary.csv"
weak_celllines_file = RESULTS_DIR / "step6a_weak_celllines.csv"
strong_celllines_file = RESULTS_DIR / "step6a_strong_celllines.csv"
model_win_count_file = RESULTS_DIR / "step6a_model_win_counts.csv"

print("Reading Step 5 files:")
print(step5_metrics_file)
print(step5_best_file)
print(step5_avg_file)

metrics_df = pd.read_csv(step5_metrics_file)
best_df = pd.read_csv(step5_best_file)
avg_df = pd.read_csv(step5_avg_file)

print("\n✅ Step 5 files loaded")
print("All metrics shape:", metrics_df.shape)
print("Best per cell line shape:", best_df.shape)
print("Average model performance shape:", avg_df.shape)

# ------------------------------------------------------------
# 1. Model win count
# ------------------------------------------------------------

model_win_counts = (
    best_df["model"]
    .value_counts()
    .reset_index()
)

model_win_counts.columns = ["model", "winning_cellline_count"]
model_win_counts["winning_percentage"] = (
    model_win_counts["winning_cellline_count"] / best_df.shape[0] * 100
)

model_win_counts.to_csv(model_win_count_file, index=False)

print("\n============================================================")
print("Model win counts")
print("============================================================")
display(model_win_counts)

# ------------------------------------------------------------
# 2. Weakest and strongest cell lines
# ------------------------------------------------------------

weak_celllines = best_df.sort_values("r2_score", ascending=True).reset_index(drop=True)
strong_celllines = best_df.sort_values("r2_score", ascending=False).reset_index(drop=True)

weak_celllines.to_csv(weak_celllines_file, index=False)
strong_celllines.to_csv(strong_celllines_file, index=False)

print("\n============================================================")
print("Weakest 15 cell lines by best R²")
print("============================================================")
display(weak_celllines[[
    "cell_line", "model", "r2_score", "rmse", "mae", "pearson_rp", "spearman_rs"
]].head(15))

print("\n============================================================")
print("Strongest 15 cell lines by best R²")
print("============================================================")
display(strong_celllines[[
    "cell_line", "model", "r2_score", "rmse", "mae", "pearson_rp", "spearman_rs"
]].head(15))

# ------------------------------------------------------------
# 3. Overall diagnostics summary
# ------------------------------------------------------------

diagnostics_summary = pd.DataFrame([{
    "cellline_count": best_df["cell_line"].nunique(),
    "model_count": metrics_df["model"].nunique(),
    "total_successful_models": metrics_df[metrics_df["status"] == "success"].shape[0],
    "mean_best_r2": best_df["r2_score"].mean(),
    "median_best_r2": best_df["r2_score"].median(),
    "min_best_r2": best_df["r2_score"].min(),
    "max_best_r2": best_df["r2_score"].max(),
    "mean_best_pearson_rp": best_df["pearson_rp"].mean(),
    "median_best_pearson_rp": best_df["pearson_rp"].median(),
    "min_best_pearson_rp": best_df["pearson_rp"].min(),
    "max_best_pearson_rp": best_df["pearson_rp"].max()
}])

diagnostics_summary.to_csv(diagnostics_file, index=False)

print("\n============================================================")
print("Step 6A diagnostics summary")
print("============================================================")
display(diagnostics_summary)

print("\n✅ Step 6A files saved:")
print("Diagnostics summary:", diagnostics_file)
print("Weak cell lines:", weak_celllines_file)
print("Strong cell lines:", strong_celllines_file)
print("Model win counts:", model_win_count_file)

print("\n✅ STEP 6A COMPLETE")

Reading Step 5 files:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_all_cellline_model_comparison.csv
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_best_model_per_cellline.csv
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_average_model_performance.csv

✅ Step 5 files loaded
All metrics shape: (240, 17)
Best per cell line shape: (60, 17)
Average model performance shape: (4, 10)

Model win counts


,model,winning_cellline_count,winning_percentage
0,CatBoost,17,28.333333
1,LightGBM,17,28.333333
2,RandomForest,15,25.000000
3,XGBoost,11,18.333333



Weakest 15 cell lines by best R²


,cell_line,model,r2_score,rmse,mae,pearson_rp,spearman_rs
0,MDA-MB-468,RandomForest,0.172254,46.158212,34.344921,0.441268,0.445078
1,LOX IMVI,RandomForest,0.222412,42.586623,30.027640,0.495517,0.509552
2,HT29,CatBoost,0.257827,43.616308,30.383854,0.525377,0.573605
3,UACC-62,LightGBM,0.278341,41.493663,31.083496,0.554659,0.507297
4,U251,XGBoost,0.289089,41.527659,25.795735,0.538263,0.458806
5,T-47D,XGBoost,0.291574,45.077081,32.723384,0.541400,0.496532
6,NCI-H522,RandomForest,0.295907,53.320803,31.095486,0.567107,0.567737
7,NCI/ADR-RES,CatBoost,0.307205,26.144349,20.450493,0.561626,0.518393
8,HOP-92,LightGBM,0.307251,48.850379,34.795505,0.565114,0.584124
9,K-562,LightGBM,0.308437,44.499773,33.443401,0.567842,0.547530



Strongest 15 cell lines by best R²


,cell_line,model,r2_score,rmse,mae,pearson_rp,spearman_rs
0,SK-MEL-5,LightGBM,0.646064,52.688494,38.029984,0.804634,0.763274
1,SW-620,LightGBM,0.593083,37.615360,28.520556,0.772715,0.682681
2,HCC-2998,LightGBM,0.565962,43.600394,31.988763,0.753638,0.663851
3,MCF7,RandomForest,0.551389,35.885049,25.958773,0.745631,0.694057
4,EKVX,XGBoost,0.513032,37.043191,26.724034,0.739068,0.649060
5,UO-31,LightGBM,0.488913,28.852973,21.092855,0.699640,0.672009
6,UACC-257,CatBoost,0.488441,30.425894,23.843307,0.705509,0.711164
7,M14,XGBoost,0.484504,43.214857,30.140675,0.696805,0.611828
8,MDA-MB-435,LightGBM,0.481028,45.904437,33.059256,0.694153,0.637014
9,KM12,RandomForest,0.480216,48.161717,33.784034,0.696539,0.719489



Step 6A diagnostics summary


,cellline_count,model_count,total_successful_models,mean_best_r2,median_best_r2,min_best_r2,max_best_r2,mean_best_pearson_rp,median_best_pearson_rp,min_best_pearson_rp,max_best_pearson_rp
0,60,4,240,0.395875,0.388419,0.172254,0.646064,0.63313,0.627854,0.441268,0.804634



✅ Step 6A files saved:
Diagnostics summary: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6a_diagnostics_summary.csv
Weak cell lines: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6a_weak_celllines.csv
Strong cell lines: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6a_strong_celllines.csv
Model win counts: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6a_model_win_counts.csv

✅ STEP 6A COMPLETE


In [16]:
# ============================================================
# STEP 6B-6E: Optimized training
# Improvements:
# 1. Enhanced interaction features
# 2. Small hyperparameter search
# 3. Early stopping for boosting models
# 4. Weighted ensemble
# 5. Compare optimized results with Step 5
# ============================================================

import json
import time
import re
import gc
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb

from scipy.stats import pearsonr, spearmanr


# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

RANDOM_STATE = 42
TEST_SIZE = 0.10
VALID_SIZE_FROM_TRAIN = 0.15

USE_INTERACTION_FEATURES = True

# Keep False for honest scientific comparison.
# Turning this True trains on clipped y but still tests on raw y.
USE_TARGET_CLIPPING = False
CLIP_LOWER_Q = 0.01
CLIP_UPPER_Q = 0.99

SAVE_MODELS = True
SAVE_PREDICTIONS = True
RUN_ENSEMBLE = True

# None = all 60 cell lines.
# For a quick test, change this to something like:
# CELL_LINES_TO_RUN = ["786-0", "A498", "HCC-2998"]
CELL_LINES_TO_RUN = None

model_matrix_file = DATA_DIR / "model_matrix.csv"
model_feature_cols_file = DATA_DIR / "model_feature_columns.json"

step5_avg_file = RESULTS_DIR / "step5_average_model_performance.csv"

optimized_metrics_file = RESULTS_DIR / "step6_optimized_all_model_comparison.csv"
optimized_best_file = RESULTS_DIR / "step6_optimized_best_model_per_cellline.csv"
optimized_avg_file = RESULTS_DIR / "step6_optimized_average_model_performance.csv"
optimized_predictions_file = RESULTS_DIR / "step6_optimized_all_predictions.csv"
validation_log_file = RESULTS_DIR / "step6_validation_tuning_log.csv"
comparison_vs_step5_file = RESULTS_DIR / "step6_optimized_vs_step5_comparison.csv"
optimized_feature_cols_file = DATA_DIR / "optimized_model_feature_columns.json"

print("Model matrix:", model_matrix_file)
print("Feature columns:", model_feature_cols_file)
print("Use interaction features:", USE_INTERACTION_FEATURES)
print("Use target clipping:", USE_TARGET_CLIPPING)
print("Cell lines to run:", "ALL" if CELL_LINES_TO_RUN is None else CELL_LINES_TO_RUN)


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def safe_filename(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    text = text.strip("_")
    return text


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        rp = np.nan
        rs = np.nan
    else:
        rp = pearsonr(y_true, y_pred)[0]
        rs = spearmanr(y_true, y_pred)[0]

    return {
        "r2_score": r2,
        "rmse": rmse,
        "mae": mae,
        "pearson_rp": rp,
        "spearman_rs": rs
    }


def build_optimized_feature_names(d1_cols, d2_cols, use_interactions=True):
    base_names = d1_cols + d2_cols

    if not use_interactions:
        return base_names

    base_drug_feature_names = [c.replace("D1_", "", 1) for c in d1_cols]

    sum_names = [f"SUM_{c}" for c in base_drug_feature_names]
    absdiff_names = [f"ABSDIFF_{c}" for c in base_drug_feature_names]
    product_names = [f"PRODUCT_{c}" for c in base_drug_feature_names]

    return base_names + sum_names + absdiff_names + product_names


def make_X_from_df(df, d1_cols, d2_cols, use_interactions=True):
    d1 = df[d1_cols].to_numpy(dtype=np.float32)
    d2 = df[d2_cols].to_numpy(dtype=np.float32)

    if not use_interactions:
        return np.hstack([d1, d2]).astype(np.float32)

    sum_features = d1 + d2
    absdiff_features = np.abs(d1 - d2)
    product_features = d1 * d2

    X = np.hstack([
        d1,
        d2,
        sum_features,
        absdiff_features,
        product_features
    ]).astype(np.float32)

    return X


def make_augmented_X_y(df, d1_cols, d2_cols, use_interactions=True, clip_bounds=None):
    d1 = df[d1_cols].to_numpy(dtype=np.float32)
    d2 = df[d2_cols].to_numpy(dtype=np.float32)

    y = df["COMBOSCORE"].to_numpy(dtype=np.float32)

    if clip_bounds is not None:
        low, high = clip_bounds
        y_train = np.clip(y, low, high).astype(np.float32)
    else:
        y_train = y

    if not use_interactions:
        X_original = np.hstack([d1, d2]).astype(np.float32)
        X_reversed = np.hstack([d2, d1]).astype(np.float32)
    else:
        sum_features = d1 + d2
        absdiff_features = np.abs(d1 - d2)
        product_features = d1 * d2

        X_original = np.hstack([
            d1,
            d2,
            sum_features,
            absdiff_features,
            product_features
        ]).astype(np.float32)

        X_reversed = np.hstack([
            d2,
            d1,
            sum_features,
            absdiff_features,
            product_features
        ]).astype(np.float32)

    X_aug = np.vstack([X_original, X_reversed]).astype(np.float32)
    y_aug = np.concatenate([y_train, y_train]).astype(np.float32)

    return X_aug, y_aug


def get_clip_bounds(train_df):
    if not USE_TARGET_CLIPPING:
        return None

    low = train_df["COMBOSCORE"].quantile(CLIP_LOWER_Q)
    high = train_df["COMBOSCORE"].quantile(CLIP_UPPER_Q)

    return float(low), float(high)


def make_model_candidates(model_name, n_features, random_state):
    rf_max_features = max(1, n_features // 3)

    if model_name == "RandomForest":
        return [
            {
                "n_estimators": 300,
                "max_features": rf_max_features,
                "min_samples_leaf": 1,
                "bootstrap": True,
                "random_state": random_state,
                "n_jobs": -1
            },
            {
                "n_estimators": 400,
                "max_features": "sqrt",
                "min_samples_leaf": 2,
                "bootstrap": True,
                "random_state": random_state,
                "n_jobs": -1
            }
        ]

    if model_name == "XGBoost":
        return [
            {
                "n_estimators": 2000,
                "max_depth": 4,
                "learning_rate": 0.02,
                "subsample": 0.90,
                "colsample_bytree": 0.85,
                "min_child_weight": 1,
                "reg_lambda": 2.0,
                "reg_alpha": 0.0,
                "objective": "reg:squarederror",
                "eval_metric": "rmse",
                "tree_method": "hist",
                "early_stopping_rounds": 60,
                "random_state": random_state,
                "n_jobs": -1
            },
            {
                "n_estimators": 1500,
                "max_depth": 5,
                "learning_rate": 0.03,
                "subsample": 0.85,
                "colsample_bytree": 0.85,
                "min_child_weight": 2,
                "reg_lambda": 3.0,
                "reg_alpha": 0.05,
                "objective": "reg:squarederror",
                "eval_metric": "rmse",
                "tree_method": "hist",
                "early_stopping_rounds": 60,
                "random_state": random_state,
                "n_jobs": -1
            }
        ]

    if model_name == "CatBoost":
        return [
            {
                "iterations": 2000,
                "depth": 5,
                "learning_rate": 0.03,
                "l2_leaf_reg": 3.0,
                "loss_function": "RMSE",
                "random_seed": random_state,
                "verbose": False,
                "allow_writing_files": False,
                "thread_count": -1
            },
            {
                "iterations": 1800,
                "depth": 6,
                "learning_rate": 0.025,
                "l2_leaf_reg": 5.0,
                "loss_function": "RMSE",
                "random_seed": random_state,
                "verbose": False,
                "allow_writing_files": False,
                "thread_count": -1
            }
        ]

    if model_name == "LightGBM":
        return [
            {
                "n_estimators": 2000,
                "learning_rate": 0.02,
                "num_leaves": 31,
                "max_depth": -1,
                "min_child_samples": 15,
                "subsample": 0.90,
                "subsample_freq": 1,
                "colsample_bytree": 0.85,
                "reg_lambda": 2.0,
                "reg_alpha": 0.0,
                "random_state": random_state,
                "n_jobs": -1,
                "verbosity": -1
            },
            {
                "n_estimators": 1600,
                "learning_rate": 0.03,
                "num_leaves": 63,
                "max_depth": -1,
                "min_child_samples": 20,
                "subsample": 0.85,
                "subsample_freq": 1,
                "colsample_bytree": 0.85,
                "reg_lambda": 3.0,
                "reg_alpha": 0.05,
                "random_state": random_state,
                "n_jobs": -1,
                "verbosity": -1
            }
        ]

    raise ValueError(f"Unknown model name: {model_name}")


def build_model(model_name, params):
    if model_name == "RandomForest":
        return RandomForestRegressor(**params)

    if model_name == "XGBoost":
        return XGBRegressor(**params)

    if model_name == "CatBoost":
        return CatBoostRegressor(**params)

    if model_name == "LightGBM":
        return LGBMRegressor(**params)

    raise ValueError(f"Unknown model name: {model_name}")


def fit_model_with_validation(model_name, model, X_train, y_train, X_val, y_val):
    best_iteration = None

    if model_name == "CatBoost":
        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            use_best_model=True,
            verbose=False
        )

        try:
            best_iteration = model.get_best_iteration()
        except Exception:
            best_iteration = None

    elif model_name == "XGBoost":
        try:
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        except TypeError:
            # Compatibility fallback for some xgboost versions
            model.fit(X_train, y_train)

        try:
            best_iteration = model.best_iteration
        except Exception:
            best_iteration = None

    elif model_name == "LightGBM":
        try:
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                eval_metric="rmse",
                callbacks=[lgb.early_stopping(60, verbose=False)]
            )
        except TypeError:
            model.fit(X_train, y_train)

        try:
            best_iteration = model.best_iteration_
        except Exception:
            best_iteration = None

    else:
        model.fit(X_train, y_train)

    return model, best_iteration


def adjust_params_for_final_fit(model_name, params, best_iteration):
    final_params = dict(params)

    if model_name == "XGBoost":
        final_params.pop("early_stopping_rounds", None)

        if best_iteration is not None and best_iteration > 0:
            final_params["n_estimators"] = int(best_iteration + 1)

    elif model_name == "CatBoost":
        if best_iteration is not None and best_iteration > 0:
            final_params["iterations"] = int(best_iteration + 1)

    elif model_name == "LightGBM":
        if best_iteration is not None and best_iteration > 0:
            final_params["n_estimators"] = int(best_iteration)

    return final_params


# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

model_matrix = pd.read_csv(model_matrix_file)

with open(model_feature_cols_file, "r") as f:
    model_feature_cols = json.load(f)

required_cols = ["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]

for col in required_cols:
    if col not in model_matrix.columns:
        raise ValueError(f"Missing required column: {col}")

missing_feature_cols = [c for c in model_feature_cols if c not in model_matrix.columns]

if missing_feature_cols:
    raise ValueError(f"Missing feature columns: {missing_feature_cols[:10]}")

d1_cols = [c for c in model_feature_cols if c.startswith("D1_")]
d2_cols = ["D2_" + c.replace("D1_", "", 1) for c in d1_cols]

if len(d1_cols) != 263:
    raise ValueError(f"Expected 263 D1 features, found {len(d1_cols)}")

if len(d2_cols) != 263:
    raise ValueError(f"Expected 263 D2 features, found {len(d2_cols)}")

for col in d2_cols:
    if col not in model_matrix.columns:
        raise ValueError(f"Missing matching D2 column: {col}")

optimized_feature_cols = build_optimized_feature_names(
    d1_cols=d1_cols,
    d2_cols=d2_cols,
    use_interactions=USE_INTERACTION_FEATURES
)

with open(optimized_feature_cols_file, "w") as f:
    json.dump(optimized_feature_cols, f, indent=2)

print("\n✅ Data loaded")
print("Model matrix shape:", model_matrix.shape)
print("Original model features:", len(model_feature_cols))
print("Optimized model features:", len(optimized_feature_cols))

cell_lines = sorted(model_matrix["CELLNAME"].unique())

if CELL_LINES_TO_RUN is not None:
    cell_lines = [c for c in cell_lines if c in CELL_LINES_TO_RUN]

print("Cell lines selected:", len(cell_lines))


# ------------------------------------------------------------
# 3. Main optimized training loop
# ------------------------------------------------------------

model_names = ["RandomForest", "XGBoost", "CatBoost", "LightGBM"]

all_metrics_rows = []
all_prediction_dfs = []
validation_log_rows = []

overall_start = time.time()

for cell_index, cell_line in enumerate(cell_lines, start=1):

    safe_cell = safe_filename(cell_line)

    print("\n\n############################################################")
    print(f"Optimized training {cell_index}/{len(cell_lines)}: {cell_line}")
    print("############################################################")

    cell_df = model_matrix[model_matrix["CELLNAME"] == cell_line].copy()
    cell_df = cell_df.reset_index(drop=True)

    print("Rows:", len(cell_df))

    train_df, test_df = train_test_split(
        cell_df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    subtrain_df, val_df = train_test_split(
        train_df,
        test_size=VALID_SIZE_FROM_TRAIN,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    subtrain_df = subtrain_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    clip_bounds = get_clip_bounds(train_df)

    print("Train rows:", len(train_df))
    print("Subtrain rows:", len(subtrain_df))
    print("Validation rows:", len(val_df))
    print("Test rows:", len(test_df))

    if clip_bounds is not None:
        print("Target clipping bounds:", clip_bounds)

    X_subtrain_aug, y_subtrain_aug = make_augmented_X_y(
        subtrain_df,
        d1_cols,
        d2_cols,
        use_interactions=USE_INTERACTION_FEATURES,
        clip_bounds=clip_bounds
    )

    X_val = make_X_from_df(
        val_df,
        d1_cols,
        d2_cols,
        use_interactions=USE_INTERACTION_FEATURES
    )

    y_val = val_df["COMBOSCORE"].to_numpy(dtype=np.float32)

    X_train_aug, y_train_aug = make_augmented_X_y(
        train_df,
        d1_cols,
        d2_cols,
        use_interactions=USE_INTERACTION_FEATURES,
        clip_bounds=clip_bounds
    )

    X_test = make_X_from_df(
        test_df,
        d1_cols,
        d2_cols,
        use_interactions=USE_INTERACTION_FEATURES
    )

    y_test = test_df["COMBOSCORE"].to_numpy(dtype=np.float32)

    print("X_subtrain_aug:", X_subtrain_aug.shape)
    print("X_val:", X_val.shape)
    print("X_train_aug:", X_train_aug.shape)
    print("X_test:", X_test.shape)

    final_test_predictions = {}
    final_model_paths = {}
    selected_validation_scores = {}
    selected_params_for_models = {}

    # --------------------------------------------------------
    # Tune and train each model
    # --------------------------------------------------------

    for model_name in model_names:

        print("\n------------------------------------------------------------")
        print(f"Tuning {model_name} for {cell_line}")
        print("------------------------------------------------------------")

        candidates = make_model_candidates(
            model_name=model_name,
            n_features=X_subtrain_aug.shape[1],
            random_state=RANDOM_STATE
        )

        best_candidate = None
        best_candidate_score = -np.inf
        best_candidate_metrics = None
        best_candidate_iteration = None

        for candidate_id, params in enumerate(candidates, start=1):

            candidate_start = time.time()

            try:
                model = build_model(model_name, params)

                model, best_iteration = fit_model_with_validation(
                    model_name=model_name,
                    model=model,
                    X_train=X_subtrain_aug,
                    y_train=y_subtrain_aug,
                    X_val=X_val,
                    y_val=y_val
                )

                val_pred = model.predict(X_val)
                val_metrics = compute_metrics(y_val, val_pred)

                candidate_time = time.time() - candidate_start
                val_score = val_metrics["r2_score"]

                validation_log_rows.append({
                    "cell_line": cell_line,
                    "model": model_name,
                    "candidate_id": candidate_id,
                    "validation_r2_score": val_metrics["r2_score"],
                    "validation_rmse": val_metrics["rmse"],
                    "validation_mae": val_metrics["mae"],
                    "validation_pearson_rp": val_metrics["pearson_rp"],
                    "validation_spearman_rs": val_metrics["spearman_rs"],
                    "best_iteration": best_iteration,
                    "candidate_time_sec": candidate_time,
                    "params": json.dumps(params),
                    "status": "success",
                    "error": ""
                })

                print(
                    f"Candidate {candidate_id}: "
                    f"val R²={val_metrics['r2_score']:.4f}, "
                    f"val Rp={val_metrics['pearson_rp']:.4f}, "
                    f"best_iter={best_iteration}"
                )

                if val_score > best_candidate_score:
                    best_candidate_score = val_score
                    best_candidate = params
                    best_candidate_metrics = val_metrics
                    best_candidate_iteration = best_iteration

            except Exception as e:
                candidate_time = time.time() - candidate_start

                validation_log_rows.append({
                    "cell_line": cell_line,
                    "model": model_name,
                    "candidate_id": candidate_id,
                    "validation_r2_score": np.nan,
                    "validation_rmse": np.nan,
                    "validation_mae": np.nan,
                    "validation_pearson_rp": np.nan,
                    "validation_spearman_rs": np.nan,
                    "best_iteration": np.nan,
                    "candidate_time_sec": candidate_time,
                    "params": json.dumps(params),
                    "status": "failed",
                    "error": str(e)
                })

                print(f"❌ Candidate {candidate_id} failed:", str(e))

        if best_candidate is None:
            print(f"❌ No successful candidate for {model_name} / {cell_line}")
            continue

        print(f"✅ Best {model_name} validation R²:", round(best_candidate_score, 4))

        # ----------------------------------------------------
        # Final fit on full 90% train set using selected params
        # ----------------------------------------------------

        final_params = adjust_params_for_final_fit(
            model_name=model_name,
            params=best_candidate,
            best_iteration=best_candidate_iteration
        )

        selected_params_for_models[model_name] = final_params
        selected_validation_scores[model_name] = best_candidate_score

        print("Final params:", final_params)

        final_start = time.time()

        try:
            final_model = build_model(model_name, final_params)

            if model_name == "CatBoost":
                final_model.fit(X_train_aug, y_train_aug, verbose=False)
            else:
                final_model.fit(X_train_aug, y_train_aug)

            train_time_sec = time.time() - final_start

            y_pred = final_model.predict(X_test)
            test_metrics = compute_metrics(y_test, y_pred)

            if SAVE_MODELS:
                model_file = MODELS_DIR / f"step6opt_{model_name.lower()}_{safe_cell}.pkl"
                joblib.dump(final_model, model_file)
            else:
                model_file = ""

            pred_df = test_df[["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]].copy()
            pred_df = pred_df.rename(columns={"COMBOSCORE": "actual_comboscore"})
            pred_df["predicted_comboscore"] = y_pred
            pred_df["prediction_error"] = (
                pred_df["actual_comboscore"] - pred_df["predicted_comboscore"]
            )
            pred_df["model"] = model_name

            if SAVE_PREDICTIONS:
                pred_file = RESULTS_DIR / f"step6opt_predictions_{model_name.lower()}_{safe_cell}.csv"
                pred_df.to_csv(pred_file, index=False)
            else:
                pred_file = ""

            all_prediction_dfs.append(pred_df)

            final_test_predictions[model_name] = y_pred
            final_model_paths[model_name] = str(model_file)

            metrics_row = {
                "cell_line": cell_line,
                "model": model_name,
                "rows_total": len(cell_df),
                "train_rows_before_augmentation": len(train_df),
                "train_rows_after_augmentation": len(train_df) * 2,
                "validation_rows": len(val_df),
                "test_rows": len(test_df),
                "input_features": len(optimized_feature_cols),
                "use_interaction_features": USE_INTERACTION_FEATURES,
                "use_target_clipping": USE_TARGET_CLIPPING,
                "clip_bounds": str(clip_bounds),
                "validation_r2_score": best_candidate_metrics["r2_score"],
                "validation_pearson_rp": best_candidate_metrics["pearson_rp"],
                "r2_score": test_metrics["r2_score"],
                "rmse": test_metrics["rmse"],
                "mae": test_metrics["mae"],
                "pearson_rp": test_metrics["pearson_rp"],
                "spearman_rs": test_metrics["spearman_rs"],
                "train_time_sec": train_time_sec,
                "model_path": str(model_file),
                "prediction_file": str(pred_file),
                "selected_params": json.dumps(final_params),
                "status": "success",
                "error": ""
            }

            all_metrics_rows.append(metrics_row)

            print(f"✅ Final {model_name} done")
            print("Test R²   :", round(test_metrics["r2_score"], 4))
            print("Test Rp   :", round(test_metrics["pearson_rp"], 4))
            print("Test RMSE :", round(test_metrics["rmse"], 4))
            print("Test MAE  :", round(test_metrics["mae"], 4))

        except Exception as e:
            train_time_sec = time.time() - final_start

            all_metrics_rows.append({
                "cell_line": cell_line,
                "model": model_name,
                "rows_total": len(cell_df),
                "train_rows_before_augmentation": len(train_df),
                "train_rows_after_augmentation": len(train_df) * 2,
                "validation_rows": len(val_df),
                "test_rows": len(test_df),
                "input_features": len(optimized_feature_cols),
                "use_interaction_features": USE_INTERACTION_FEATURES,
                "use_target_clipping": USE_TARGET_CLIPPING,
                "clip_bounds": str(clip_bounds),
                "validation_r2_score": best_candidate_score,
                "validation_pearson_rp": np.nan,
                "r2_score": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "pearson_rp": np.nan,
                "spearman_rs": np.nan,
                "train_time_sec": train_time_sec,
                "model_path": "",
                "prediction_file": "",
                "selected_params": json.dumps(final_params),
                "status": "failed",
                "error": str(e)
            })

            print(f"❌ Final {model_name} failed:", str(e))

    # --------------------------------------------------------
    # Weighted ensemble
    # --------------------------------------------------------

    if RUN_ENSEMBLE and len(final_test_predictions) >= 2:

        print("\n------------------------------------------------------------")
        print(f"Building weighted ensemble for {cell_line}")
        print("------------------------------------------------------------")

        ensemble_model_names = list(final_test_predictions.keys())

        raw_weights = np.array([
            max(selected_validation_scores.get(m, 0.0), 0.0001)
            for m in ensemble_model_names
        ], dtype=float)

        if raw_weights.sum() == 0:
            weights = np.ones(len(ensemble_model_names)) / len(ensemble_model_names)
        else:
            weights = raw_weights / raw_weights.sum()

        y_pred_ensemble = np.zeros_like(y_test, dtype=float)

        for model_name, weight in zip(ensemble_model_names, weights):
            y_pred_ensemble += weight * final_test_predictions[model_name]

        ensemble_metrics = compute_metrics(y_test, y_pred_ensemble)

        ensemble_config = {
            "cell_line": cell_line,
            "ensemble_type": "validation_r2_weighted_average",
            "feature_count": len(optimized_feature_cols),
            "use_interaction_features": USE_INTERACTION_FEATURES,
            "use_target_clipping": USE_TARGET_CLIPPING,
            "models": [
                {
                    "model": model_name,
                    "weight": float(weight),
                    "model_path": final_model_paths.get(model_name, ""),
                    "validation_r2_score": float(selected_validation_scores.get(model_name, np.nan)),
                    "selected_params": selected_params_for_models.get(model_name, {})
                }
                for model_name, weight in zip(ensemble_model_names, weights)
            ]
        }

        ensemble_config_file = MODELS_DIR / f"step6opt_ensemble_config_{safe_cell}.json"

        with open(ensemble_config_file, "w") as f:
            json.dump(ensemble_config, f, indent=2)

        pred_df = test_df[["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]].copy()
        pred_df = pred_df.rename(columns={"COMBOSCORE": "actual_comboscore"})
        pred_df["predicted_comboscore"] = y_pred_ensemble
        pred_df["prediction_error"] = (
            pred_df["actual_comboscore"] - pred_df["predicted_comboscore"]
        )
        pred_df["model"] = "Ensemble"

        if SAVE_PREDICTIONS:
            pred_file = RESULTS_DIR / f"step6opt_predictions_ensemble_{safe_cell}.csv"
            pred_df.to_csv(pred_file, index=False)
        else:
            pred_file = ""

        all_prediction_dfs.append(pred_df)

        all_metrics_rows.append({
            "cell_line": cell_line,
            "model": "Ensemble",
            "rows_total": len(cell_df),
            "train_rows_before_augmentation": len(train_df),
            "train_rows_after_augmentation": len(train_df) * 2,
            "validation_rows": len(val_df),
            "test_rows": len(test_df),
            "input_features": len(optimized_feature_cols),
            "use_interaction_features": USE_INTERACTION_FEATURES,
            "use_target_clipping": USE_TARGET_CLIPPING,
            "clip_bounds": str(clip_bounds),
            "validation_r2_score": np.nan,
            "validation_pearson_rp": np.nan,
            "r2_score": ensemble_metrics["r2_score"],
            "rmse": ensemble_metrics["rmse"],
            "mae": ensemble_metrics["mae"],
            "pearson_rp": ensemble_metrics["pearson_rp"],
            "spearman_rs": ensemble_metrics["spearman_rs"],
            "train_time_sec": 0.0,
            "model_path": str(ensemble_config_file),
            "prediction_file": str(pred_file),
            "selected_params": json.dumps(ensemble_config),
            "status": "success",
            "error": ""
        })

        print("✅ Ensemble done")
        print("Test R²   :", round(ensemble_metrics["r2_score"], 4))
        print("Test Rp   :", round(ensemble_metrics["pearson_rp"], 4))
        print("Weights:")
        for m, w in zip(ensemble_model_names, weights):
            print(f"  {m}: {w:.4f}")

    # --------------------------------------------------------
    # Save progress after each cell line
    # --------------------------------------------------------

    progress_metrics = pd.DataFrame(all_metrics_rows)
    progress_metrics.to_csv(optimized_metrics_file, index=False)

    progress_validation = pd.DataFrame(validation_log_rows)
    progress_validation.to_csv(validation_log_file, index=False)

    if SAVE_PREDICTIONS and len(all_prediction_dfs) > 0:
        progress_predictions = pd.concat(all_prediction_dfs, axis=0, ignore_index=True)
        progress_predictions.to_csv(optimized_predictions_file, index=False)

    print("\n✅ Progress saved after:", cell_line)

    del X_subtrain_aug, y_subtrain_aug, X_val, y_val, X_train_aug, y_train_aug, X_test, y_test
    gc.collect()


# ------------------------------------------------------------
# 4. Final optimized reports
# ------------------------------------------------------------

total_time_sec = time.time() - overall_start

metrics_df = pd.DataFrame(all_metrics_rows)

metrics_df = metrics_df.sort_values(
    by=["cell_line", "r2_score"],
    ascending=[True, False]
).reset_index(drop=True)

metrics_df.to_csv(optimized_metrics_file, index=False)

validation_log_df = pd.DataFrame(validation_log_rows)
validation_log_df.to_csv(validation_log_file, index=False)

successful_metrics = metrics_df[metrics_df["status"] == "success"].copy()

best_per_cellline = (
    successful_metrics
    .sort_values(["cell_line", "r2_score"], ascending=[True, False])
    .groupby("cell_line", as_index=False)
    .first()
)

best_per_cellline.to_csv(optimized_best_file, index=False)

average_model_perf = (
    successful_metrics
    .groupby("model", as_index=False)
    .agg(
        mean_r2_score=("r2_score", "mean"),
        median_r2_score=("r2_score", "median"),
        mean_rmse=("rmse", "mean"),
        mean_mae=("mae", "mean"),
        mean_pearson_rp=("pearson_rp", "mean"),
        median_pearson_rp=("pearson_rp", "median"),
        mean_spearman_rs=("spearman_rs", "mean"),
        successful_cellline_count=("cell_line", "nunique")
    )
    .sort_values("mean_r2_score", ascending=False)
    .reset_index(drop=True)
)

average_model_perf.to_csv(optimized_avg_file, index=False)

if SAVE_PREDICTIONS and len(all_prediction_dfs) > 0:
    all_predictions_df = pd.concat(all_prediction_dfs, axis=0, ignore_index=True)
    all_predictions_df.to_csv(optimized_predictions_file, index=False)


# ------------------------------------------------------------
# 5. Compare optimized results with Step 5
# ------------------------------------------------------------

if step5_avg_file.exists():
    old_avg = pd.read_csv(step5_avg_file)

    old_avg_small = old_avg[[
        "model",
        "mean_r2_score",
        "mean_pearson_rp",
        "mean_rmse",
        "mean_mae"
    ]].copy()

    old_avg_small = old_avg_small.rename(columns={
        "mean_r2_score": "step5_mean_r2_score",
        "mean_pearson_rp": "step5_mean_pearson_rp",
        "mean_rmse": "step5_mean_rmse",
        "mean_mae": "step5_mean_mae"
    })

    new_avg_small = average_model_perf[[
        "model",
        "mean_r2_score",
        "mean_pearson_rp",
        "mean_rmse",
        "mean_mae"
    ]].copy()

    new_avg_small = new_avg_small.rename(columns={
        "mean_r2_score": "step6_mean_r2_score",
        "mean_pearson_rp": "step6_mean_pearson_rp",
        "mean_rmse": "step6_mean_rmse",
        "mean_mae": "step6_mean_mae"
    })

    comparison = new_avg_small.merge(
        old_avg_small,
        on="model",
        how="left"
    )

    comparison["r2_improvement"] = (
        comparison["step6_mean_r2_score"] - comparison["step5_mean_r2_score"]
    )

    comparison["pearson_rp_improvement"] = (
        comparison["step6_mean_pearson_rp"] - comparison["step5_mean_pearson_rp"]
    )

    comparison["rmse_change"] = (
        comparison["step6_mean_rmse"] - comparison["step5_mean_rmse"]
    )

    comparison["mae_change"] = (
        comparison["step6_mean_mae"] - comparison["step5_mean_mae"]
    )

    comparison = comparison.sort_values(
        "step6_mean_r2_score",
        ascending=False
    ).reset_index(drop=True)

    comparison.to_csv(comparison_vs_step5_file, index=False)

else:
    comparison = pd.DataFrame()


# ------------------------------------------------------------
# 6. Display final results
# ------------------------------------------------------------

print("\n\n============================================================")
print("STEP 6 OPTIMIZED TRAINING COMPLETE")
print("============================================================")
print("Total time:", round(total_time_sec / 60, 2), "minutes")
print("Metrics shape:", metrics_df.shape)
print("Best per cell line shape:", best_per_cellline.shape)
print("Average performance shape:", average_model_perf.shape)

print("\n============================================================")
print("Optimized average model performance")
print("============================================================")
display(average_model_perf)

print("\n============================================================")
print("Optimized best model per cell line")
print("============================================================")
display(best_per_cellline[[
    "cell_line",
    "model",
    "r2_score",
    "rmse",
    "mae",
    "pearson_rp",
    "spearman_rs"
]].head(25))

if not comparison.empty:
    print("\n============================================================")
    print("Step 6 optimized vs Step 5 comparison")
    print("============================================================")
    display(comparison)

print("\n✅ Saved files:")
print("Optimized all metrics:", optimized_metrics_file)
print("Optimized best per cell line:", optimized_best_file)
print("Optimized average performance:", optimized_avg_file)
print("Validation tuning log:", validation_log_file)
print("Optimized feature columns:", optimized_feature_cols_file)

if SAVE_PREDICTIONS:
    print("Optimized all predictions:", optimized_predictions_file)

if step5_avg_file.exists():
    print("Step 6 vs Step 5 comparison:", comparison_vs_step5_file)

print("\n✅ STEP 6B-6E COMPLETE")

Model matrix: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_matrix.csv
Feature columns: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_feature_columns.json
Use interaction features: True
Use target clipping: False
Cell lines to run: ALL

✅ Data loaded
Model matrix shape: (145212, 530)
Original model features: 526
Optimized model features: 1315
Cell lines selected: 60


############################################################
Optimized training 1/60: 786-0
############################################################
Rows: 2439
Train rows: 2195
Subtrain rows: 1865
Validation rows: 330
Test rows: 244
X_subtrain_aug: (3730, 1315)
X_val: (330, 1315)
X_train_aug: (4390, 1315)
X_test: (244, 1315)

------------------------------------------------------------
Tuning RandomForest for 786-0
------------------------------------------------------------
Candidate 1: val R²=0.2115, val Rp=0.4694, best_iter=None
Candidate 2: val R²=0.2483, val Rp=0.5065, best_iter=None
✅ B

,model,mean_r2_score,median_r2_score,mean_rmse,mean_mae,mean_pearson_rp,median_pearson_rp,mean_spearman_rs,successful_cellline_count
0,Ensemble,0.349286,0.331969,45.205532,31.938798,0.594884,0.584792,0.567905,60
1,CatBoost,0.343876,0.326250,45.367548,32.180643,0.589321,0.575225,0.558804,60
2,LightGBM,0.338379,0.327334,45.511725,32.245922,0.586419,0.575543,0.557741,60
3,RandomForest,0.329384,0.321733,45.935113,32.321226,0.577265,0.574597,0.557798,60
4,XGBoost,0.326540,0.327404,46.042657,32.567569,0.575634,0.576129,0.547719,60



Optimized best model per cell line


,cell_line,model,r2_score,rmse,mae,pearson_rp,spearman_rs
0,786-0,Ensemble,0.338190,30.972495,24.487742,0.582985,0.583115
1,A498,CatBoost,0.419703,33.815690,24.066474,0.648217,0.614327
2,A549/ATCC,Ensemble,0.327368,30.988813,22.987009,0.575516,0.553553
3,ACHN,Ensemble,0.352194,43.048212,32.630230,0.607984,0.529531
4,BT-549,RandomForest,0.347572,47.701759,34.230973,0.600092,0.559476
5,CAKI-1,CatBoost,0.337545,44.936901,32.112641,0.589265,0.576140
6,CCRF-CEM,RandomForest,0.355473,58.727661,44.087126,0.602035,0.455698
7,COLO 205,Ensemble,0.305453,48.326331,36.055594,0.560457,0.527053
8,DU-145,RandomForest,0.343134,47.823873,28.466214,0.593024,0.593460
9,EKVX,CatBoost,0.507784,37.242271,26.955552,0.730381,0.631394



Step 6 optimized vs Step 5 comparison


,model,step6_mean_r2_score,step6_mean_pearson_rp,step6_mean_rmse,step6_mean_mae,step5_mean_r2_score,step5_mean_pearson_rp,step5_mean_rmse,step5_mean_mae,r2_improvement,pearson_rp_improvement,rmse_change,mae_change
0,Ensemble,0.349286,0.594884,45.205532,31.938798,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CatBoost,0.343876,0.589321,45.367548,32.180643,0.374818,0.617905,44.221958,31.615502,-0.030941,-0.028584,1.145590,0.565140
2,LightGBM,0.338379,0.586419,45.511725,32.245922,0.370570,0.612983,44.290690,31.475354,-0.032190,-0.026563,1.221035,0.770568
3,RandomForest,0.329384,0.577265,45.935113,32.321226,0.371234,0.612695,44.317994,31.315135,-0.041850,-0.035429,1.617118,1.006091
4,XGBoost,0.326540,0.575634,46.042657,32.567569,0.369660,0.611927,44.422604,31.646120,-0.043120,-0.036293,1.620053,0.921449



✅ Saved files:
Optimized all metrics: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_optimized_all_model_comparison.csv
Optimized best per cell line: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_optimized_best_model_per_cellline.csv
Optimized average performance: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_optimized_average_model_performance.csv
Validation tuning log: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_validation_tuning_log.csv
Optimized feature columns: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\optimized_model_feature_columns.json
Optimized all predictions: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_optimized_all_predictions.csv
Step 6 vs Step 5 comparison: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_optimized_vs_step5_comparison.csv

✅ STEP 6B-6E COMPLETE


# Continue here

In [17]:
# ============================================================
# STEP 6 FINAL: Train final best model for each cell line
# Uses Step 5 best model selection
# Uses original 526 features only
# Trains on 100% data + drug-order augmentation
# Saves final 60 models
# ============================================================

import json
import time
import re
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor


# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

RANDOM_STATE = 42

model_matrix_file = DATA_DIR / "model_matrix.csv"
model_feature_cols_file = DATA_DIR / "model_feature_columns.json"
step5_best_file = RESULTS_DIR / "step5_best_model_per_cellline.csv"

final_summary_file = RESULTS_DIR / "step6_final_model_summary.csv"
final_registry_file = RESULTS_DIR / "step6_final_model_registry.csv"
final_feature_cols_file = DATA_DIR / "step6_final_model_feature_columns.json"

print("Model matrix file:", model_matrix_file)
print("Feature columns file:", model_feature_cols_file)
print("Step 5 best model file:", step5_best_file)


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def safe_filename(text):
    """
    Converts cell-line names into safe filenames.
    Example:
        786-0 -> 786_0
        A549/ATCC -> A549_ATCC
    """
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    text = text.strip("_")
    return text


def augment_reverse_drug_order(df, d1_cols, d2_cols):
    """
    Final training augmentation.

    Original:
        NSC1 + NSC2
        D1 features + D2 features

    Reversed:
        NSC2 + NSC1
        D2 features + D1 features

    COMBOSCORE remains same.
    """
    original = df.copy()
    reversed_df = df.copy()

    # Swap NSC IDs
    reversed_df[["NSC1", "NSC2"]] = reversed_df[["NSC2", "NSC1"]].to_numpy()

    # Swap D1 and D2 features
    temp_d1 = reversed_df[d1_cols].copy()
    reversed_df[d1_cols] = reversed_df[d2_cols].to_numpy()
    reversed_df[d2_cols] = temp_d1.to_numpy()

    final_train_df = pd.concat(
        [original, reversed_df],
        axis=0,
        ignore_index=True
    )

    return final_train_df


def make_final_model(model_name, random_state, n_features):
    """
    Uses the same model settings as Step 5.
    This keeps final models consistent with official Step 5 evaluation.
    """
    rf_max_features = max(1, n_features // 3)

    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=250,
            max_features=rf_max_features,
            random_state=random_state,
            n_jobs=-1
        )

    elif model_name == "XGBoost":
        return XGBRegressor(
            n_estimators=700,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist"
        )

    elif model_name == "CatBoost":
        return CatBoostRegressor(
            iterations=700,
            depth=6,
            learning_rate=0.03,
            loss_function="RMSE",
            random_seed=random_state,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        )

    elif model_name == "LightGBM":
        return LGBMRegressor(
            n_estimators=700,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1
        )

    else:
        raise ValueError(f"Unknown model name: {model_name}")


# ------------------------------------------------------------
# 2. Load files
# ------------------------------------------------------------

model_matrix = pd.read_csv(model_matrix_file)

with open(model_feature_cols_file, "r") as f:
    model_feature_cols = json.load(f)

step5_best = pd.read_csv(step5_best_file)

print("\n✅ Files loaded")
print("Model matrix shape:", model_matrix.shape)
print("Feature count:", len(model_feature_cols))
print("Step 5 best shape:", step5_best.shape)


# ------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------

required_matrix_cols = ["NSC1", "NSC2", "CELLNAME", "COMBOSCORE"]

for col in required_matrix_cols:
    if col not in model_matrix.columns:
        raise ValueError(f"Missing required column in model_matrix: {col}")

required_best_cols = ["cell_line", "model", "r2_score", "pearson_rp"]

for col in required_best_cols:
    if col not in step5_best.columns:
        raise ValueError(f"Missing required column in Step 5 best file: {col}")

missing_feature_cols = [c for c in model_feature_cols if c not in model_matrix.columns]

if missing_feature_cols:
    raise ValueError(f"Missing feature columns in model_matrix: {missing_feature_cols[:10]}")

if len(model_feature_cols) != 526:
    raise ValueError(f"Expected 526 model features, found {len(model_feature_cols)}")

print("\n✅ Required columns validated")
print("Using original 526 Step 5 features only")


# ------------------------------------------------------------
# 4. Prepare D1 and D2 feature groups
# ------------------------------------------------------------

d1_cols = [c for c in model_feature_cols if c.startswith("D1_")]
d2_cols = ["D2_" + c.replace("D1_", "", 1) for c in d1_cols]

if len(d1_cols) != 263:
    raise ValueError(f"Expected 263 D1 features, found {len(d1_cols)}")

if len(d2_cols) != 263:
    raise ValueError(f"Expected 263 D2 features, found {len(d2_cols)}")

for col in d2_cols:
    if col not in model_feature_cols:
        raise ValueError(f"Missing D2 feature column: {col}")

print("\n✅ Feature groups ready")
print("D1 feature count:", len(d1_cols))
print("D2 feature count:", len(d2_cols))


# ------------------------------------------------------------
# 5. Save final feature columns for later prediction use
# ------------------------------------------------------------

with open(final_feature_cols_file, "w") as f:
    json.dump(model_feature_cols, f, indent=2)

print("\n✅ Final feature columns saved:")
print(final_feature_cols_file)


# ------------------------------------------------------------
# 6. Train final selected model for each cell line
# ------------------------------------------------------------

summary_rows = []
registry_rows = []

overall_start = time.time()

step5_best_sorted = step5_best.sort_values("cell_line").reset_index(drop=True)

print("\n============================================================")
print("FINAL STEP 6 TRAINING STARTED")
print("============================================================")
print("Cell lines to train:", step5_best_sorted["cell_line"].nunique())

for idx, row in step5_best_sorted.iterrows():

    cell_line = row["cell_line"]
    selected_model_name = row["model"]
    safe_cell = safe_filename(cell_line)

    print("\n############################################################")
    print(f"Final model {idx + 1}/{len(step5_best_sorted)}")
    print("Cell line:", cell_line)
    print("Selected model from Step 5:", selected_model_name)
    print("Step 5 R²:", round(row["r2_score"], 4))
    print("Step 5 Rp:", round(row["pearson_rp"], 4))
    print("############################################################")

    start_time = time.time()

    try:
        # Filter all available data for this cell line
        cell_df = model_matrix[model_matrix["CELLNAME"] == cell_line].copy()
        cell_df = cell_df.reset_index(drop=True)

        if cell_df.empty:
            raise ValueError(f"No rows found for cell line: {cell_line}")

        original_rows = len(cell_df)

        # Augment full data
        final_train_df = augment_reverse_drug_order(
            cell_df,
            d1_cols=d1_cols,
            d2_cols=d2_cols
        )

        augmented_rows = len(final_train_df)

        # Prepare X/y
        X_final = final_train_df[model_feature_cols].to_numpy(dtype=np.float32)
        y_final = final_train_df["COMBOSCORE"].to_numpy(dtype=np.float32)

        # Create selected final model
        final_model = make_final_model(
            model_name=selected_model_name,
            random_state=RANDOM_STATE,
            n_features=X_final.shape[1]
        )

        # Train final model on 100% data
        final_model.fit(X_final, y_final)

        train_time_sec = time.time() - start_time

        # Save final model
        final_model_file = MODELS_DIR / f"final_step6_{selected_model_name.lower()}_{safe_cell}.pkl"
        joblib.dump(final_model, final_model_file)

        # Add summary row
        summary_rows.append({
            "cell_line": cell_line,
            "selected_model": selected_model_name,
            "original_rows": original_rows,
            "augmented_training_rows": augmented_rows,
            "input_features": len(model_feature_cols),
            "step5_r2_score": row["r2_score"],
            "step5_rmse": row.get("rmse", np.nan),
            "step5_mae": row.get("mae", np.nan),
            "step5_pearson_rp": row["pearson_rp"],
            "step5_spearman_rs": row.get("spearman_rs", np.nan),
            "final_model_path": str(final_model_file),
            "train_time_sec": train_time_sec,
            "status": "success",
            "error": ""
        })

        # Add registry row for app/prediction usage
        registry_rows.append({
            "cell_line": cell_line,
            "safe_cell_line": safe_cell,
            "selected_model": selected_model_name,
            "model_path": str(final_model_file),
            "feature_columns_file": str(final_feature_cols_file),
            "input_features": len(model_feature_cols)
        })

        print("✅ Final model trained and saved")
        print("Original rows:", original_rows)
        print("Augmented rows:", augmented_rows)
        print("Input features:", X_final.shape[1])
        print("Train time:", round(train_time_sec, 2), "sec")
        print("Saved to:", final_model_file)

    except Exception as e:
        train_time_sec = time.time() - start_time

        summary_rows.append({
            "cell_line": cell_line,
            "selected_model": selected_model_name,
            "original_rows": np.nan,
            "augmented_training_rows": np.nan,
            "input_features": len(model_feature_cols),
            "step5_r2_score": row.get("r2_score", np.nan),
            "step5_rmse": row.get("rmse", np.nan),
            "step5_mae": row.get("mae", np.nan),
            "step5_pearson_rp": row.get("pearson_rp", np.nan),
            "step5_spearman_rs": row.get("spearman_rs", np.nan),
            "final_model_path": "",
            "train_time_sec": train_time_sec,
            "status": "failed",
            "error": str(e)
        })

        print("❌ Final model failed")
        print("Error:", str(e))

    # Save progress after every cell line
    pd.DataFrame(summary_rows).to_csv(final_summary_file, index=False)
    pd.DataFrame(registry_rows).to_csv(final_registry_file, index=False)


# ------------------------------------------------------------
# 7. Save final reports
# ------------------------------------------------------------

total_time_sec = time.time() - overall_start

final_summary = pd.DataFrame(summary_rows)
final_registry = pd.DataFrame(registry_rows)

final_summary.to_csv(final_summary_file, index=False)
final_registry.to_csv(final_registry_file, index=False)

successful_count = (final_summary["status"] == "success").sum()
failed_count = (final_summary["status"] == "failed").sum()

print("\n\n============================================================")
print("STEP 6 FINAL COMPLETE")
print("============================================================")
print("Total final models expected:", len(step5_best_sorted))
print("Successful final models:", successful_count)
print("Failed final models:", failed_count)
print("Total training time:", round(total_time_sec / 60, 2), "minutes")

print("\nFinal model summary:")
display(final_summary.head(20))

print("\nModel counts:")
display(final_summary["selected_model"].value_counts().reset_index().rename(
    columns={"index": "selected_model", "selected_model": "count"}
))

print("\n✅ Saved files:")
print("Final model summary:", final_summary_file)
print("Final model registry:", final_registry_file)
print("Final feature columns:", final_feature_cols_file)

print("\n✅ Final models saved in:")
print(MODELS_DIR)

print("\nIMPORTANT:")
print("Step 5 metrics remain the official evaluation.")
print("Step 6 final models are trained on 100% data for final prediction/deployment.")

Model matrix file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_matrix.csv
Feature columns file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\model_feature_columns.json
Step 5 best model file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_best_model_per_cellline.csv

✅ Files loaded
Model matrix shape: (145212, 530)
Feature count: 526
Step 5 best shape: (60, 17)

✅ Required columns validated
Using original 526 Step 5 features only

✅ Feature groups ready
D1 feature count: 263
D2 feature count: 263

✅ Final feature columns saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\step6_final_model_feature_columns.json

FINAL STEP 6 TRAINING STARTED
Cell lines to train: 60

############################################################
Final model 1/60
Cell line: 786-0
Selected model from Step 5: CatBoost
Step 5 R²: 0.3419
Step 5 Rp: 0.5858
############################################################
✅ Final model trained and saved
Original rows: 2439

,cell_line,selected_model,original_rows,augmented_training_rows,input_features,step5_r2_score,step5_rmse,step5_mae,step5_pearson_rp,step5_spearman_rs,final_model_path,train_time_sec,status,error
0,786-0,CatBoost,2439,4878,526,0.341898,30.885618,24.368193,0.585791,0.587811,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,4.934116,success,
1,A498,CatBoost,2441,4882,526,0.447089,33.008112,23.161057,0.673602,0.641828,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,5.457619,success,
2,A549/ATCC,LightGBM,2439,4878,526,0.395637,29.374134,21.848186,0.629480,0.584564,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,2.635016,success,
3,ACHN,RandomForest,2434,4868,526,0.371066,42.416528,31.872216,0.623041,0.564880,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,11.857502,success,
4,BT-549,RandomForest,2431,4862,526,0.336038,48.121570,33.925945,0.585430,0.542594,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,11.691208,success,
5,CAKI-1,LightGBM,2436,4872,526,0.378873,43.512602,30.896801,0.622691,0.597871,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,2.558956,success,
6,CCRF-CEM,RandomForest,2399,4798,526,0.386014,57.319395,42.582848,0.627295,0.513332,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,11.938006,success,
7,COLO 205,CatBoost,2426,4852,526,0.341847,47.043135,35.993181,0.588019,0.532671,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,4.963355,success,
8,DU-145,XGBoost,2427,4854,526,0.434345,44.379482,27.544412,0.662433,0.616230,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,2.672150,success,
9,EKVX,XGBoost,2439,4878,526,0.513032,37.043191,26.724034,0.739068,0.649060,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,2.732620,success,



Model counts:


,count,count
0,CatBoost,17
1,LightGBM,17
2,RandomForest,15
3,XGBoost,11



✅ Saved files:
Final model summary: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_final_model_summary.csv
Final model registry: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_final_model_registry.csv
Final feature columns: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\step6_final_model_feature_columns.json

✅ Final models saved in:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\models

IMPORTANT:
Step 5 metrics remain the official evaluation.
Step 6 final models are trained on 100% data for final prediction/deployment.


In [18]:
# ============================================================
# STEP 7: Single ComboScore prediction
# Input: NSC1, NSC2, CELLNAME
# Output: Predicted ComboScore
# ============================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Change only these 3 values for prediction
# ------------------------------------------------------------

INPUT_NSC1 = 740
INPUT_NSC2 = 752
INPUT_CELLNAME = "786-0"


# ------------------------------------------------------------
# 2. Required files
# ------------------------------------------------------------

drug_features_file = DATA_DIR / "drug_features.csv"
feature_columns_file = DATA_DIR / "step6_final_model_feature_columns.json"
model_registry_file = RESULTS_DIR / "step6_final_model_registry.csv"

print("Drug features file:", drug_features_file)
print("Feature columns file:", feature_columns_file)
print("Model registry file:", model_registry_file)


# ------------------------------------------------------------
# 3. Load saved files
# ------------------------------------------------------------

drug_features = pd.read_csv(drug_features_file)
model_registry = pd.read_csv(model_registry_file)

with open(feature_columns_file, "r") as f:
    model_feature_columns = json.load(f)

print("\n✅ Files loaded")
print("Drug features shape:", drug_features.shape)
print("Model registry shape:", model_registry.shape)
print("Model feature count:", len(model_feature_columns))


# ------------------------------------------------------------
# 4. Basic validation
# ------------------------------------------------------------

if "NSC" not in drug_features.columns:
    raise ValueError("drug_features.csv must contain NSC column.")

if len(model_feature_columns) != 526:
    raise ValueError(f"Expected 526 model features, found {len(model_feature_columns)}")

drug_features["NSC"] = drug_features["NSC"].astype(int)

feature_cols_263 = [c for c in drug_features.columns if c != "NSC"]

if len(feature_cols_263) != 263:
    raise ValueError(f"Expected 263 drug features, found {len(feature_cols_263)}")

available_nscs = set(drug_features["NSC"].tolist())

if int(INPUT_NSC1) not in available_nscs:
    raise ValueError(f"NSC1 {INPUT_NSC1} not found in drug_features.csv")

if int(INPUT_NSC2) not in available_nscs:
    raise ValueError(f"NSC2 {INPUT_NSC2} not found in drug_features.csv")

if INPUT_CELLNAME not in set(model_registry["cell_line"].tolist()):
    raise ValueError(f"Cell line {INPUT_CELLNAME} not found in final model registry.")

print("\n✅ Input validated")
print("NSC1:", INPUT_NSC1)
print("NSC2:", INPUT_NSC2)
print("CELLNAME:", INPUT_CELLNAME)


# ------------------------------------------------------------
# 5. Helper function to create 526 features
# ------------------------------------------------------------

def create_pair_features(nsc1, nsc2):
    """
    Creates one model input row:
    D1_feat_0 ... D1_feat_262
    D2_feat_0 ... D2_feat_262
    """

    nsc1 = int(nsc1)
    nsc2 = int(nsc2)

    drug1 = drug_features[drug_features["NSC"] == nsc1].iloc[0]
    drug2 = drug_features[drug_features["NSC"] == nsc2].iloc[0]

    row = {}

    for col in feature_cols_263:
        row["D1_" + col] = float(drug1[col])
        row["D2_" + col] = float(drug2[col])

    X = pd.DataFrame([row])

    # Make sure column order is exactly same as training
    X = X[model_feature_columns]

    return X


# ------------------------------------------------------------
# 6. Load correct final model for selected cell line
# ------------------------------------------------------------

registry_row = model_registry[model_registry["cell_line"] == INPUT_CELLNAME].iloc[0]

selected_model = registry_row["selected_model"]
model_path = Path(registry_row["model_path"])

# Fallback in case saved absolute path differs
if not model_path.exists():
    model_path = MODELS_DIR / Path(registry_row["model_path"]).name

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")

model = joblib.load(model_path)

print("\n✅ Final model loaded")
print("Cell line:", INPUT_CELLNAME)
print("Model used:", selected_model)
print("Model path:", model_path)


# ------------------------------------------------------------
# 7. Predict both drug orders and average them
# ------------------------------------------------------------

X_order_1 = create_pair_features(INPUT_NSC1, INPUT_NSC2)
X_order_2 = create_pair_features(INPUT_NSC2, INPUT_NSC1)

prediction_order_1 = float(model.predict(X_order_1)[0])
prediction_order_2 = float(model.predict(X_order_2)[0])

final_prediction = (prediction_order_1 + prediction_order_2) / 2


# ------------------------------------------------------------
# 8. Show final result
# ------------------------------------------------------------

prediction_result = pd.DataFrame([{
    "NSC1": INPUT_NSC1,
    "NSC2": INPUT_NSC2,
    "CELLNAME": INPUT_CELLNAME,
    "model_used": selected_model,
    "prediction_NSC1_to_NSC2": prediction_order_1,
    "prediction_NSC2_to_NSC1": prediction_order_2,
    "final_predicted_COMBOSCORE": final_prediction
}])

print("\n============================================================")
print("FINAL PREDICTION RESULT")
print("============================================================")
display(prediction_result)

print("\n✅ STEP 7 SINGLE PREDICTION COMPLETE")

Drug features file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv
Feature columns file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\step6_final_model_feature_columns.json
Model registry file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_final_model_registry.csv

✅ Files loaded
Drug features shape: (100, 264)
Model registry shape: (60, 6)
Model feature count: 526

✅ Input validated
NSC1: 740
NSC2: 752
CELLNAME: 786-0

✅ Final model loaded
Cell line: 786-0
Model used: CatBoost
Model path: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\models\final_step6_catboost_786_0.pkl

FINAL PREDICTION RESULT


,NSC1,NSC2,CELLNAME,model_used,prediction_NSC1_to_NSC2,prediction_NSC2_to_NSC1,final_predicted_COMBOSCORE
0,740,752,786-0,CatBoost,-40.261658,-39.059509,-39.660584



✅ STEP 7 SINGLE PREDICTION COMPLETE


In [19]:
# ============================================================
# STEP 7A: Create predictions folder and test one official test entry
# Uses Step 5 test predictions
# This is fair testing because Step 5 used untouched 10% test data
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Create predictions folder
# ------------------------------------------------------------

PREDICTIONS_DIR = PROJECT_DIR / "predictions"
PREDICTIONS_DIR.mkdir(exist_ok=True)

print("✅ Predictions folder ready:")
print(PREDICTIONS_DIR)


# ------------------------------------------------------------
# 2. Input files from Step 5
# ------------------------------------------------------------

step5_all_predictions_file = RESULTS_DIR / "step5_all_cellline_all_model_predictions.csv"
step5_best_model_file = RESULTS_DIR / "step5_best_model_per_cellline.csv"

print("\nReading files:")
print("All Step 5 predictions:", step5_all_predictions_file)
print("Step 5 best models    :", step5_best_model_file)


# ------------------------------------------------------------
# 3. Output files inside predictions folder
# ------------------------------------------------------------

official_test_dataset_file = PREDICTIONS_DIR / "step5_official_test_dataset.csv"
best_model_test_predictions_file = PREDICTIONS_DIR / "step5_best_model_test_predictions.csv"
single_prediction_file = PREDICTIONS_DIR / "single_test_entry_prediction.csv"


# ------------------------------------------------------------
# 4. Load files
# ------------------------------------------------------------

all_predictions = pd.read_csv(step5_all_predictions_file)
best_models = pd.read_csv(step5_best_model_file)

print("\n✅ Files loaded")
print("All predictions shape:", all_predictions.shape)
print("Best models shape:", best_models.shape)

print("\nAll prediction columns:")
print(all_predictions.columns.tolist())

print("\nBest model columns:")
print(best_models.columns.tolist())


# ------------------------------------------------------------
# 5. Validate columns
# ------------------------------------------------------------

required_prediction_cols = [
    "NSC1",
    "NSC2",
    "CELLNAME",
    "actual_comboscore",
    "predicted_comboscore",
    "prediction_error",
    "model"
]

for col in required_prediction_cols:
    if col not in all_predictions.columns:
        raise ValueError(f"Missing column in Step 5 predictions file: {col}")

required_best_cols = ["cell_line", "model"]

for col in required_best_cols:
    if col not in best_models.columns:
        raise ValueError(f"Missing column in Step 5 best model file: {col}")

print("\n✅ Required columns validated")


# ------------------------------------------------------------
# 6. Create clean official test dataset
# ------------------------------------------------------------
# Step 5 prediction file has repeated rows because each test entry
# has predictions from RandomForest, XGBoost, CatBoost, and LightGBM.
# Here we remove those model duplicates and keep only actual test entries.

official_test_dataset = (
    all_predictions[["NSC1", "NSC2", "CELLNAME", "actual_comboscore"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

official_test_dataset.to_csv(official_test_dataset_file, index=False)

print("\n============================================================")
print("Official Step 5 test dataset created")
print("============================================================")
print("Shape:", official_test_dataset.shape)
print("Saved to:", official_test_dataset_file)

display(official_test_dataset.head())


# ------------------------------------------------------------
# 7. Keep only best-model prediction for each test row
# ------------------------------------------------------------
# For each CELLNAME, Step 5 selected one best model.
# We merge that information and keep only that model prediction.

best_models_small = best_models[["cell_line", "model"]].copy()
best_models_small = best_models_small.rename(columns={
    "cell_line": "CELLNAME",
    "model": "best_model_for_cellline"
})

predictions_with_best = all_predictions.merge(
    best_models_small,
    on="CELLNAME",
    how="left"
)

best_model_test_predictions = predictions_with_best[
    predictions_with_best["model"] == predictions_with_best["best_model_for_cellline"]
].copy()

best_model_test_predictions = best_model_test_predictions.reset_index(drop=True)

best_model_test_predictions.to_csv(best_model_test_predictions_file, index=False)

print("\n============================================================")
print("Best-model test predictions created")
print("============================================================")
print("Shape:", best_model_test_predictions.shape)
print("Saved to:", best_model_test_predictions_file)

display(best_model_test_predictions.head())


# ------------------------------------------------------------
# 8. Select one test entry to check
# ------------------------------------------------------------
# Change this number to test another row.
# Example: 0 means first test row, 10 means 11th test row.

TEST_ROW_INDEX = 0

if TEST_ROW_INDEX < 0 or TEST_ROW_INDEX >= len(official_test_dataset):
    raise ValueError(
        f"TEST_ROW_INDEX must be between 0 and {len(official_test_dataset) - 1}"
    )

test_entry = official_test_dataset.iloc[TEST_ROW_INDEX]

test_nsc1 = test_entry["NSC1"]
test_nsc2 = test_entry["NSC2"]
test_cellline = test_entry["CELLNAME"]
test_actual = test_entry["actual_comboscore"]

print("\n============================================================")
print("Selected test entry")
print("============================================================")
print("TEST_ROW_INDEX:", TEST_ROW_INDEX)
print("NSC1:", test_nsc1)
print("NSC2:", test_nsc2)
print("CELLNAME:", test_cellline)
print("Actual ComboScore:", test_actual)


# ------------------------------------------------------------
# 9. Get prediction for this test entry using best model
# ------------------------------------------------------------

single_prediction = best_model_test_predictions[
    (best_model_test_predictions["NSC1"] == test_nsc1) &
    (best_model_test_predictions["NSC2"] == test_nsc2) &
    (best_model_test_predictions["CELLNAME"] == test_cellline)
].copy()

if single_prediction.empty:
    raise ValueError("No best-model prediction found for selected test entry.")

single_prediction = single_prediction.reset_index(drop=True)

single_prediction.to_csv(single_prediction_file, index=False)

print("\n============================================================")
print("Single test-entry prediction result")
print("============================================================")
display(single_prediction)

print("\n✅ Single prediction saved to:")
print(single_prediction_file)


# ------------------------------------------------------------
# 10. Show simple final result
# ------------------------------------------------------------

row = single_prediction.iloc[0]

print("\n============================================================")
print("FINAL SIMPLE RESULT")
print("============================================================")
print("NSC1:", row["NSC1"])
print("NSC2:", row["NSC2"])
print("CELLNAME:", row["CELLNAME"])
print("Best model used:", row["model"])
print("Actual ComboScore:", row["actual_comboscore"])
print("Predicted ComboScore:", row["predicted_comboscore"])
print("Prediction error:", row["prediction_error"])

print("\n✅ STEP 7A COMPLETE")
print("All prediction outputs saved inside:")
print(PREDICTIONS_DIR)

✅ Predictions folder ready:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions

Reading files:
All Step 5 predictions: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_all_cellline_all_model_predictions.csv
Step 5 best models    : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step5_best_model_per_cellline.csv

✅ Files loaded
All predictions shape: (58180, 7)
Best models shape: (60, 17)

All prediction columns:
['NSC1', 'NSC2', 'CELLNAME', 'actual_comboscore', 'predicted_comboscore', 'prediction_error', 'model']

Best model columns:
['cell_line', 'model', 'rows_total', 'train_rows_before_augmentation', 'train_rows_after_augmentation', 'test_rows', 'input_features', 'r2_score', 'rmse', 'mae', 'pearson_rp', 'spearman_rs', 'train_time_sec', 'model_path', 'prediction_file', 'status', 'error']

✅ Required columns validated

Official Step 5 test dataset created
Shape: (14545, 4)
Saved to: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\step5_officia

,NSC1,NSC2,CELLNAME,actual_comboscore
0,8806,715055,786-0,-20.0
1,8806,606869,786-0,-17.0
2,18509,715055,786-0,-27.0
3,85998,756645,786-0,-58.0
4,13875,26271,786-0,-40.0



Best-model test predictions created
Shape: (14545, 8)
Saved to: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\step5_best_model_test_predictions.csv


,NSC1,NSC2,CELLNAME,actual_comboscore,predicted_comboscore,prediction_error,model,best_model_for_cellline
0,8806,715055,786-0,-20.0,-17.003711,-2.996289,CatBoost,CatBoost
1,8806,606869,786-0,-17.0,-25.337913,8.337913,CatBoost,CatBoost
2,18509,715055,786-0,-27.0,-19.149363,-7.850637,CatBoost,CatBoost
3,85998,756645,786-0,-58.0,-11.215723,-46.784277,CatBoost,CatBoost
4,13875,26271,786-0,-40.0,-31.340848,-8.659152,CatBoost,CatBoost



Selected test entry
TEST_ROW_INDEX: 0
NSC1: 8806
NSC2: 715055
CELLNAME: 786-0
Actual ComboScore: -20.0

Single test-entry prediction result


,NSC1,NSC2,CELLNAME,actual_comboscore,predicted_comboscore,prediction_error,model,best_model_for_cellline
0,8806,715055,786-0,-20.0,-17.003711,-2.996289,CatBoost,CatBoost



✅ Single prediction saved to:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\single_test_entry_prediction.csv

FINAL SIMPLE RESULT
NSC1: 8806
NSC2: 715055
CELLNAME: 786-0
Best model used: CatBoost
Actual ComboScore: -20.0
Predicted ComboScore: -17.003710685606578
Prediction error: -2.9962893143934224

✅ STEP 7A COMPLETE
All prediction outputs saved inside:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions


In [20]:
# ============================================================
# STEP 7B: Final model prediction for custom input
# Uses Step 6 final saved models
# Input: NSC1, NSC2, CELLNAME
# Output: Final predicted ComboScore
# ============================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Folder setup
# ------------------------------------------------------------

try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = Path(".").resolve()

try:
    DATA_DIR
except NameError:
    DATA_DIR = PROJECT_DIR / "data"

try:
    RESULTS_DIR
except NameError:
    RESULTS_DIR = PROJECT_DIR / "results"

try:
    MODELS_DIR
except NameError:
    MODELS_DIR = PROJECT_DIR / "models"

PREDICTIONS_DIR = PROJECT_DIR / "predictions"
PREDICTIONS_DIR.mkdir(exist_ok=True)

print("✅ Predictions folder ready:")
print(PREDICTIONS_DIR)


# ------------------------------------------------------------
# 2. Change only these 3 values for custom prediction
# ------------------------------------------------------------

INPUT_NSC1 = 740
INPUT_NSC2 = 752
INPUT_CELLNAME = "786-0"


# ------------------------------------------------------------
# 3. Required files
# ------------------------------------------------------------

drug_features_file = DATA_DIR / "drug_features.csv"
feature_columns_file = DATA_DIR / "step6_final_model_feature_columns.json"
model_registry_file = RESULTS_DIR / "step6_final_model_registry.csv"

single_prediction_file = PREDICTIONS_DIR / "final_model_single_prediction.csv"
prediction_history_file = PREDICTIONS_DIR / "final_model_prediction_history.csv"

print("\nInput files:")
print("Drug features file :", drug_features_file)
print("Feature columns file:", feature_columns_file)
print("Model registry file:", model_registry_file)

print("\nOutput files:")
print("Single prediction file:", single_prediction_file)
print("Prediction history file:", prediction_history_file)


# ------------------------------------------------------------
# 4. Load files
# ------------------------------------------------------------

drug_features = pd.read_csv(drug_features_file)
model_registry = pd.read_csv(model_registry_file)

with open(feature_columns_file, "r") as f:
    model_feature_columns = json.load(f)

print("\n✅ Files loaded")
print("Drug features shape:", drug_features.shape)
print("Model registry shape:", model_registry.shape)
print("Feature count:", len(model_feature_columns))


# ------------------------------------------------------------
# 5. Validate files and input
# ------------------------------------------------------------

if "NSC" not in drug_features.columns:
    raise ValueError("drug_features.csv must contain NSC column.")

required_registry_cols = ["cell_line", "selected_model", "model_path"]

for col in required_registry_cols:
    if col not in model_registry.columns:
        raise ValueError(f"Missing column in model registry: {col}")

if len(model_feature_columns) != 526:
    raise ValueError(f"Expected 526 model features, found {len(model_feature_columns)}")

drug_features["NSC"] = drug_features["NSC"].astype(int)

feature_cols_263 = [c for c in drug_features.columns if c != "NSC"]

if len(feature_cols_263) != 263:
    raise ValueError(f"Expected 263 drug features, found {len(feature_cols_263)}")

INPUT_NSC1 = int(INPUT_NSC1)
INPUT_NSC2 = int(INPUT_NSC2)

available_nscs = set(drug_features["NSC"].tolist())

if INPUT_NSC1 not in available_nscs:
    raise ValueError(f"NSC1 {INPUT_NSC1} not found in drug_features.csv")

if INPUT_NSC2 not in available_nscs:
    raise ValueError(f"NSC2 {INPUT_NSC2} not found in drug_features.csv")

available_cell_lines = set(model_registry["cell_line"].tolist())

if INPUT_CELLNAME not in available_cell_lines:
    print("\nAvailable cell lines:")
    print(sorted(list(available_cell_lines)))
    raise ValueError(f"Cell line '{INPUT_CELLNAME}' not found in final model registry.")

print("\n✅ Input validated")
print("NSC1:", INPUT_NSC1)
print("NSC2:", INPUT_NSC2)
print("CELLNAME:", INPUT_CELLNAME)


# ------------------------------------------------------------
# 6. Function to create 526 model features
# ------------------------------------------------------------

def create_pair_features(nsc1, nsc2):
    """
    Creates one prediction row with 526 features:
    D1_feat_0 ... D1_feat_262
    D2_feat_0 ... D2_feat_262
    """

    nsc1 = int(nsc1)
    nsc2 = int(nsc2)

    drug1 = drug_features[drug_features["NSC"] == nsc1].iloc[0]
    drug2 = drug_features[drug_features["NSC"] == nsc2].iloc[0]

    row = {}

    for col in feature_cols_263:
        row["D1_" + col] = float(drug1[col])
        row["D2_" + col] = float(drug2[col])

    X = pd.DataFrame([row])

    # Very important: use the same column order used during training
    X = X[model_feature_columns]

    return X


# ------------------------------------------------------------
# 7. Load correct final model for selected cell line
# ------------------------------------------------------------

registry_row = model_registry[model_registry["cell_line"] == INPUT_CELLNAME].iloc[0]

selected_model = registry_row["selected_model"]
model_path = Path(registry_row["model_path"])

# If absolute path does not exist, use local models folder fallback
if not model_path.exists():
    model_path = MODELS_DIR / Path(registry_row["model_path"]).name

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")

model = joblib.load(model_path)

print("\n✅ Final model loaded")
print("Cell line:", INPUT_CELLNAME)
print("Model used:", selected_model)
print("Model path:", model_path)


# ------------------------------------------------------------
# 8. Predict both drug orders
# ------------------------------------------------------------
# Because drug pair order should not matter, we predict both:
# NSC1 -> NSC2
# NSC2 -> NSC1
# Then we average them.

X_order_1 = create_pair_features(INPUT_NSC1, INPUT_NSC2)
X_order_2 = create_pair_features(INPUT_NSC2, INPUT_NSC1)

prediction_order_1 = float(model.predict(X_order_1)[0])
prediction_order_2 = float(model.predict(X_order_2)[0])

final_predicted_comboscore = (prediction_order_1 + prediction_order_2) / 2


# ------------------------------------------------------------
# 9. Save prediction result
# ------------------------------------------------------------

prediction_result = pd.DataFrame([{
    "NSC1": INPUT_NSC1,
    "NSC2": INPUT_NSC2,
    "CELLNAME": INPUT_CELLNAME,
    "model_used": selected_model,
    "prediction_NSC1_to_NSC2": prediction_order_1,
    "prediction_NSC2_to_NSC1": prediction_order_2,
    "final_predicted_COMBOSCORE": final_predicted_comboscore,
    "model_path": str(model_path)
}])

# Save latest single prediction
prediction_result.to_csv(single_prediction_file, index=False)

# Also append to prediction history
if prediction_history_file.exists():
    old_history = pd.read_csv(prediction_history_file)
    new_history = pd.concat([old_history, prediction_result], ignore_index=True)
else:
    new_history = prediction_result.copy()

new_history.to_csv(prediction_history_file, index=False)


# ------------------------------------------------------------
# 10. Show final output
# ------------------------------------------------------------

print("\n============================================================")
print("FINAL MODEL PREDICTION RESULT")
print("============================================================")

display(prediction_result)

print("\nSimple result:")
print("NSC1:", INPUT_NSC1)
print("NSC2:", INPUT_NSC2)
print("CELLNAME:", INPUT_CELLNAME)
print("Model used:", selected_model)
print("Prediction NSC1 -> NSC2:", round(prediction_order_1, 4))
print("Prediction NSC2 -> NSC1:", round(prediction_order_2, 4))
print("Final predicted ComboScore:", round(final_predicted_comboscore, 4))

print("\n✅ Saved latest prediction to:")
print(single_prediction_file)

print("\n✅ Added prediction to history file:")
print(prediction_history_file)

print("\n✅ STEP 7B COMPLETE")

✅ Predictions folder ready:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions

Input files:
Drug features file : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv
Feature columns file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\step6_final_model_feature_columns.json
Model registry file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_final_model_registry.csv

Output files:
Single prediction file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\final_model_single_prediction.csv
Prediction history file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\final_model_prediction_history.csv

✅ Files loaded
Drug features shape: (100, 264)
Model registry shape: (60, 6)
Feature count: 526

✅ Input validated
NSC1: 740
NSC2: 752
CELLNAME: 786-0

✅ Final model loaded
Cell line: 786-0
Model used: CatBoost
Model path: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\models\final_step6_catboost_786_0.pkl

FINAL MODEL PREDICTI

,NSC1,NSC2,CELLNAME,model_used,prediction_NSC1_to_NSC2,prediction_NSC2_to_NSC1,final_predicted_COMBOSCORE,model_path
0,740,752,786-0,CatBoost,-40.261658,-39.059509,-39.660584,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...



Simple result:
NSC1: 740
NSC2: 752
CELLNAME: 786-0
Model used: CatBoost
Prediction NSC1 -> NSC2: -40.2617
Prediction NSC2 -> NSC1: -39.0595
Final predicted ComboScore: -39.6606

✅ Saved latest prediction to:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\final_model_single_prediction.csv

✅ Added prediction to history file:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\final_model_prediction_history.csv

✅ STEP 7B COMPLETE


In [21]:
# ============================================================
# STEP 7C: Batch prediction using final Step 6 models
# Input CSV : predictions/batch_prediction_input.csv
# Output CSV: predictions/batch_prediction_output.csv
# ============================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Folder setup
# ------------------------------------------------------------

try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = Path(".").resolve()

try:
    DATA_DIR
except NameError:
    DATA_DIR = PROJECT_DIR / "data"

try:
    RESULTS_DIR
except NameError:
    RESULTS_DIR = PROJECT_DIR / "results"

try:
    MODELS_DIR
except NameError:
    MODELS_DIR = PROJECT_DIR / "models"

PREDICTIONS_DIR = PROJECT_DIR / "predictions"
PREDICTIONS_DIR.mkdir(exist_ok=True)

print("✅ Predictions folder ready:")
print(PREDICTIONS_DIR)


# ------------------------------------------------------------
# 2. Required input/output files
# ------------------------------------------------------------

drug_features_file = DATA_DIR / "drug_features.csv"
feature_columns_file = DATA_DIR / "step6_final_model_feature_columns.json"
model_registry_file = RESULTS_DIR / "step6_final_model_registry.csv"

batch_input_file = PREDICTIONS_DIR / "batch_prediction_input.csv"
batch_output_file = PREDICTIONS_DIR / "batch_prediction_output.csv"

print("\nInput files:")
print("Drug features file :", drug_features_file)
print("Feature columns file:", feature_columns_file)
print("Model registry file:", model_registry_file)

print("\nBatch files:")
print("Batch input file :", batch_input_file)
print("Batch output file:", batch_output_file)


# ------------------------------------------------------------
# 3. Load required files
# ------------------------------------------------------------

drug_features = pd.read_csv(drug_features_file)
model_registry = pd.read_csv(model_registry_file)

with open(feature_columns_file, "r") as f:
    model_feature_columns = json.load(f)

print("\n✅ Required files loaded")
print("Drug features shape:", drug_features.shape)
print("Model registry shape:", model_registry.shape)
print("Feature count:", len(model_feature_columns))


# ------------------------------------------------------------
# 4. Validate loaded files
# ------------------------------------------------------------

if "NSC" not in drug_features.columns:
    raise ValueError("drug_features.csv must contain NSC column.")

required_registry_cols = ["cell_line", "selected_model", "model_path"]

for col in required_registry_cols:
    if col not in model_registry.columns:
        raise ValueError(f"Missing column in model registry: {col}")

if len(model_feature_columns) != 526:
    raise ValueError(f"Expected 526 model features, found {len(model_feature_columns)}")

drug_features["NSC"] = drug_features["NSC"].astype(int)

feature_cols_263 = [c for c in drug_features.columns if c != "NSC"]

if len(feature_cols_263) != 263:
    raise ValueError(f"Expected 263 drug features, found {len(feature_cols_263)}")

drug_feature_map = drug_features.set_index("NSC")

available_nscs = set(drug_feature_map.index.tolist())
available_cell_lines = set(model_registry["cell_line"].tolist())

print("\n✅ File validation complete")
print("Available drugs:", len(available_nscs))
print("Available cell lines:", len(available_cell_lines))


# ------------------------------------------------------------
# 5. Create sample batch input file if it does not exist
# ------------------------------------------------------------
# First time, this creates a sample input CSV.
# Later, you can edit this CSV and rerun this same cell.

if not batch_input_file.exists():
    sample_nscs = sorted(list(available_nscs))[:4]
    sample_cell_lines = sorted(list(available_cell_lines))[:3]

    sample_batch = pd.DataFrame([
        {
            "NSC1": sample_nscs[0],
            "NSC2": sample_nscs[1],
            "CELLNAME": sample_cell_lines[0]
        },
        {
            "NSC1": sample_nscs[0],
            "NSC2": sample_nscs[2],
            "CELLNAME": sample_cell_lines[1]
        },
        {
            "NSC1": sample_nscs[1],
            "NSC2": sample_nscs[3],
            "CELLNAME": sample_cell_lines[2]
        }
    ])

    sample_batch.to_csv(batch_input_file, index=False)

    print("\n✅ Sample batch input file created:")
    print(batch_input_file)

    print("\nYou can edit this file with your own NSC1, NSC2, CELLNAME values and rerun this cell.")

else:
    print("\n✅ Batch input file already exists:")
    print(batch_input_file)


# ------------------------------------------------------------
# 6. Load batch input file
# ------------------------------------------------------------

batch_input = pd.read_csv(batch_input_file)

required_batch_cols = ["NSC1", "NSC2", "CELLNAME"]

for col in required_batch_cols:
    if col not in batch_input.columns:
        raise ValueError(f"Missing column in batch input file: {col}")

batch_input = batch_input[required_batch_cols].copy()
batch_input["NSC1"] = batch_input["NSC1"].astype(int)
batch_input["NSC2"] = batch_input["NSC2"].astype(int)
batch_input["CELLNAME"] = batch_input["CELLNAME"].astype(str)

print("\n============================================================")
print("Batch input loaded")
print("============================================================")
print("Rows to predict:", len(batch_input))
display(batch_input.head(10))


# ------------------------------------------------------------
# 7. Helper functions
# ------------------------------------------------------------

loaded_models = {}

def get_model_for_cellline(cell_line):
    """
    Loads correct final model for a cell line.
    Uses cache, so same model is not loaded again and again.
    """

    if cell_line in loaded_models:
        return loaded_models[cell_line]

    registry_row = model_registry[model_registry["cell_line"] == cell_line].iloc[0]

    selected_model = registry_row["selected_model"]
    model_path = Path(registry_row["model_path"])

    # Fallback if absolute path changed
    if not model_path.exists():
        model_path = MODELS_DIR / Path(registry_row["model_path"]).name

    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    model = joblib.load(model_path)

    loaded_models[cell_line] = {
        "model": model,
        "selected_model": selected_model,
        "model_path": model_path
    }

    return loaded_models[cell_line]


def create_pair_features(nsc1, nsc2):
    """
    Creates 526 features:
    D1_feat_0 ... D1_feat_262
    D2_feat_0 ... D2_feat_262
    """

    drug1 = drug_feature_map.loc[int(nsc1)]
    drug2 = drug_feature_map.loc[int(nsc2)]

    row = {}

    for col in feature_cols_263:
        row["D1_" + col] = float(drug1[col])
        row["D2_" + col] = float(drug2[col])

    X = pd.DataFrame([row])

    # Keep exact training column order
    X = X[model_feature_columns]

    return X


def predict_single_pair(nsc1, nsc2, cell_line):
    """
    Predicts one drug pair for one cell line.
    Predicts both drug orders and averages the result.
    """

    if int(nsc1) not in available_nscs:
        raise ValueError(f"NSC1 {nsc1} not found in drug_features.csv")

    if int(nsc2) not in available_nscs:
        raise ValueError(f"NSC2 {nsc2} not found in drug_features.csv")

    if cell_line not in available_cell_lines:
        raise ValueError(f"Cell line '{cell_line}' not found in final model registry")

    model_info = get_model_for_cellline(cell_line)

    model = model_info["model"]
    selected_model = model_info["selected_model"]
    model_path = model_info["model_path"]

    X_order_1 = create_pair_features(nsc1, nsc2)
    X_order_2 = create_pair_features(nsc2, nsc1)

    pred_order_1 = float(model.predict(X_order_1)[0])
    pred_order_2 = float(model.predict(X_order_2)[0])

    final_prediction = (pred_order_1 + pred_order_2) / 2

    return {
        "model_used": selected_model,
        "prediction_NSC1_to_NSC2": pred_order_1,
        "prediction_NSC2_to_NSC1": pred_order_2,
        "final_predicted_COMBOSCORE": final_prediction,
        "model_path": str(model_path)
    }


# ------------------------------------------------------------
# 8. Run batch predictions
# ------------------------------------------------------------

results = []

print("\n============================================================")
print("Running batch predictions")
print("============================================================")

for idx, row in batch_input.iterrows():

    nsc1 = row["NSC1"]
    nsc2 = row["NSC2"]
    cell_line = row["CELLNAME"]

    print(f"Predicting row {idx + 1}/{len(batch_input)}: NSC1={nsc1}, NSC2={nsc2}, CELLNAME={cell_line}")

    result_row = {
        "row_index": idx,
        "NSC1": nsc1,
        "NSC2": nsc2,
        "CELLNAME": cell_line
    }

    try:
        pred_result = predict_single_pair(nsc1, nsc2, cell_line)

        result_row.update(pred_result)
        result_row["status"] = "success"
        result_row["error"] = ""

        print("  ✅ Predicted ComboScore:", round(pred_result["final_predicted_COMBOSCORE"], 4))

    except Exception as e:
        result_row["model_used"] = ""
        result_row["prediction_NSC1_to_NSC2"] = np.nan
        result_row["prediction_NSC2_to_NSC1"] = np.nan
        result_row["final_predicted_COMBOSCORE"] = np.nan
        result_row["model_path"] = ""
        result_row["status"] = "failed"
        result_row["error"] = str(e)

        print("  ❌ Failed:", str(e))

    results.append(result_row)


# ------------------------------------------------------------
# 9. Save batch prediction output
# ------------------------------------------------------------

batch_output = pd.DataFrame(results)
batch_output.to_csv(batch_output_file, index=False)

print("\n============================================================")
print("BATCH PREDICTION COMPLETE")
print("============================================================")
print("Total rows:", len(batch_output))
print("Successful predictions:", (batch_output["status"] == "success").sum())
print("Failed predictions:", (batch_output["status"] == "failed").sum())

print("\n✅ Batch prediction output saved to:")
print(batch_output_file)

display(batch_output)

print("\n✅ STEP 7C COMPLETE")

✅ Predictions folder ready:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions

Input files:
Drug features file : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\drug_features.csv
Feature columns file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\data\step6_final_model_feature_columns.json
Model registry file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\results\step6_final_model_registry.csv

Batch files:
Batch input file : C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\batch_prediction_input.csv
Batch output file: C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\batch_prediction_output.csv

✅ Required files loaded
Drug features shape: (100, 264)
Model registry shape: (60, 6)
Feature count: 526

✅ File validation complete
Available drugs: 100
Available cell lines: 60

✅ Sample batch input file created:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\batch_prediction_input.csv

You can edit this file with your own NSC1, NSC2, CELL

,NSC1,NSC2,CELLNAME
0,740,750,786-0
1,740,752,A498
2,750,755,A549/ATCC



Running batch predictions
Predicting row 1/3: NSC1=740, NSC2=750, CELLNAME=786-0
  ✅ Predicted ComboScore: -16.8756
Predicting row 2/3: NSC1=740, NSC2=752, CELLNAME=A498
  ✅ Predicted ComboScore: -27.4895
Predicting row 3/3: NSC1=750, NSC2=755, CELLNAME=A549/ATCC
  ✅ Predicted ComboScore: -27.1511

BATCH PREDICTION COMPLETE
Total rows: 3
Successful predictions: 3
Failed predictions: 0

✅ Batch prediction output saved to:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\predictions\batch_prediction_output.csv


,row_index,NSC1,NSC2,CELLNAME,model_used,prediction_NSC1_to_NSC2,prediction_NSC2_to_NSC1,final_predicted_COMBOSCORE,model_path,status,error
0,0,740,750,786-0,CatBoost,-15.484548,-18.266708,-16.875628,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
1,1,740,752,A498,CatBoost,-29.466952,-25.511975,-27.489464,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,
2,2,750,755,A549/ATCC,LightGBM,-24.163685,-30.138568,-27.151127,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,



✅ STEP 7C COMPLETE


In [22]:
# ============================================================
# STEP 8: Final project summary files
# Creates clean final summary files for report + Flask/Codex use
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Folder setup
# ------------------------------------------------------------

try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = Path(".").resolve()

try:
    DATA_DIR
except NameError:
    DATA_DIR = PROJECT_DIR / "data"

try:
    RESULTS_DIR
except NameError:
    RESULTS_DIR = PROJECT_DIR / "results"

try:
    MODELS_DIR
except NameError:
    MODELS_DIR = PROJECT_DIR / "models"

PREDICTIONS_DIR = PROJECT_DIR / "predictions"
FINAL_SUMMARY_DIR = PROJECT_DIR / "final_project_summary"

PREDICTIONS_DIR.mkdir(exist_ok=True)
FINAL_SUMMARY_DIR.mkdir(exist_ok=True)

print("✅ Final summary folder ready:")
print(FINAL_SUMMARY_DIR)


# ------------------------------------------------------------
# 2. Important input files
# ------------------------------------------------------------

files = {
    "combo_score": DATA_DIR / "comboscore.csv",
    "drug_features": DATA_DIR / "drug_features.csv",
    "model_matrix": DATA_DIR / "model_matrix.csv",
    "feature_columns": DATA_DIR / "step6_final_model_feature_columns.json",

    "step5_all_model_comparison": RESULTS_DIR / "step5_all_cellline_model_comparison.csv",
    "step5_best_model_per_cellline": RESULTS_DIR / "step5_best_model_per_cellline.csv",
    "step5_average_model_performance": RESULTS_DIR / "step5_average_model_performance.csv",

    "step6_final_model_summary": RESULTS_DIR / "step6_final_model_summary.csv",
    "step6_final_model_registry": RESULTS_DIR / "step6_final_model_registry.csv",

    "step7a_official_test_dataset": PREDICTIONS_DIR / "step5_official_test_dataset.csv",
    "step7a_best_model_test_predictions": PREDICTIONS_DIR / "step5_best_model_test_predictions.csv",
    "step7b_single_prediction": PREDICTIONS_DIR / "final_model_single_prediction.csv",
    "step7c_batch_input": PREDICTIONS_DIR / "batch_prediction_input.csv",
    "step7c_batch_output": PREDICTIONS_DIR / "batch_prediction_output.csv",
}

print("\nChecking important files:")
for name, path in files.items():
    print(f"{name:35s} -> {'FOUND' if path.exists() else 'MISSING'}")


# ------------------------------------------------------------
# 3. Load main files
# ------------------------------------------------------------

drug_features = pd.read_csv(files["drug_features"])
step5_avg = pd.read_csv(files["step5_average_model_performance"])
step5_best = pd.read_csv(files["step5_best_model_per_cellline"])
step6_summary = pd.read_csv(files["step6_final_model_summary"])
step6_registry = pd.read_csv(files["step6_final_model_registry"])

with open(files["feature_columns"], "r") as f:
    final_feature_columns = json.load(f)

print("\n✅ Main files loaded")
print("Drug features shape:", drug_features.shape)
print("Step 5 average performance shape:", step5_avg.shape)
print("Step 5 best models shape:", step5_best.shape)
print("Step 6 final summary shape:", step6_summary.shape)
print("Step 6 registry shape:", step6_registry.shape)
print("Final feature count:", len(final_feature_columns))


# ------------------------------------------------------------
# 4. Output file paths
# ------------------------------------------------------------

final_model_performance_file = FINAL_SUMMARY_DIR / "final_model_performance_summary.csv"
final_best_models_file = FINAL_SUMMARY_DIR / "final_best_models_per_cellline.csv"
final_model_counts_file = FINAL_SUMMARY_DIR / "final_model_counts.csv"
final_available_inputs_file = FINAL_SUMMARY_DIR / "final_available_inputs.csv"
final_flask_manifest_file = FINAL_SUMMARY_DIR / "final_flask_assets_manifest.csv"
final_pipeline_files_file = FINAL_SUMMARY_DIR / "final_pipeline_files_summary.csv"
final_project_readme_file = FINAL_SUMMARY_DIR / "final_project_readme.txt"
final_cell_lines_file = FINAL_SUMMARY_DIR / "available_cell_lines.txt"
final_drugs_file = FINAL_SUMMARY_DIR / "available_drug_nscs.txt"


# ------------------------------------------------------------
# 5. Final model performance summary
# ------------------------------------------------------------
# This is the official evaluation result from Step 5.
# Step 6 final models are trained on 100% data, so Step 5 metrics remain official.

final_model_performance = step5_avg.copy()

# Sort by best mean R2 if column exists
if "mean_r2_score" in final_model_performance.columns:
    final_model_performance = final_model_performance.sort_values(
        "mean_r2_score",
        ascending=False
    ).reset_index(drop=True)

final_model_performance.to_csv(final_model_performance_file, index=False)

print("\n✅ Final model performance summary saved:")
print(final_model_performance_file)
display(final_model_performance)


# ------------------------------------------------------------
# 6. Final best model per cell line
# ------------------------------------------------------------

final_best_models = step5_best.copy()

# Add final model path from Step 6 registry
registry_small = step6_registry[["cell_line", "selected_model", "model_path"]].copy()

final_best_models = final_best_models.merge(
    registry_small,
    on="cell_line",
    how="left"
)

final_best_models.to_csv(final_best_models_file, index=False)

print("\n✅ Final best models per cell line saved:")
print(final_best_models_file)
display(final_best_models.head(20))


# ------------------------------------------------------------
# 7. Final model counts
# ------------------------------------------------------------

final_model_counts = (
    step6_summary["selected_model"]
    .value_counts()
    .reset_index()
)

final_model_counts.columns = ["model", "final_model_count"]

final_model_counts["percentage"] = (
    final_model_counts["final_model_count"] / final_model_counts["final_model_count"].sum() * 100
).round(2)

final_model_counts.to_csv(final_model_counts_file, index=False)

print("\n✅ Final model counts saved:")
print(final_model_counts_file)
display(final_model_counts)


# ------------------------------------------------------------
# 8. Available inputs for prediction
# ------------------------------------------------------------

available_drugs = sorted(drug_features["NSC"].astype(int).unique().tolist())
available_cell_lines = sorted(step6_registry["cell_line"].astype(str).unique().tolist())

final_available_inputs = pd.DataFrame([
    {
        "item": "available_drugs",
        "count": len(available_drugs),
        "source_file": str(files["drug_features"])
    },
    {
        "item": "available_cell_lines",
        "count": len(available_cell_lines),
        "source_file": str(files["step6_final_model_registry"])
    },
    {
        "item": "final_model_features",
        "count": len(final_feature_columns),
        "source_file": str(files["feature_columns"])
    },
    {
        "item": "final_models",
        "count": len(list(MODELS_DIR.glob("final_step6_*.pkl"))),
        "source_file": str(MODELS_DIR)
    }
])

final_available_inputs.to_csv(final_available_inputs_file, index=False)

with open(final_cell_lines_file, "w") as f:
    for cell_line in available_cell_lines:
        f.write(str(cell_line) + "\n")

with open(final_drugs_file, "w") as f:
    for nsc in available_drugs:
        f.write(str(nsc) + "\n")

print("\n✅ Available inputs saved:")
print(final_available_inputs_file)
print(final_cell_lines_file)
print(final_drugs_file)
display(final_available_inputs)


# ------------------------------------------------------------
# 9. Flask assets manifest
# ------------------------------------------------------------
# These are the files Codex/Flask needs for real prediction.

final_model_files = sorted(list(MODELS_DIR.glob("final_step6_*.pkl")))

manifest_rows = [
    {
        "asset_type": "drug_features",
        "required_for_flask": "yes",
        "path": str(files["drug_features"]),
        "description": "Contains 263 features for each available drug NSC."
    },
    {
        "asset_type": "feature_columns",
        "required_for_flask": "yes",
        "path": str(files["feature_columns"]),
        "description": "Exact 526 feature column order used by final models."
    },
    {
        "asset_type": "model_registry",
        "required_for_flask": "yes",
        "path": str(files["step6_final_model_registry"]),
        "description": "Maps each cell line to its selected final model path."
    },
    {
        "asset_type": "models_folder",
        "required_for_flask": "yes",
        "path": str(MODELS_DIR),
        "description": "Folder containing final trained .pkl models."
    },
    {
        "asset_type": "single_prediction_example",
        "required_for_flask": "no",
        "path": str(files["step7b_single_prediction"]),
        "description": "Example output from one custom prediction."
    },
    {
        "asset_type": "batch_prediction_example",
        "required_for_flask": "no",
        "path": str(files["step7c_batch_output"]),
        "description": "Example output from batch prediction."
    }
]

for model_file in final_model_files:
    manifest_rows.append({
        "asset_type": "final_model_file",
        "required_for_flask": "yes",
        "path": str(model_file),
        "description": "One final trained model file for a specific cell line."
    })

final_flask_manifest = pd.DataFrame(manifest_rows)
final_flask_manifest.to_csv(final_flask_manifest_file, index=False)

print("\n✅ Flask assets manifest saved:")
print(final_flask_manifest_file)
display(final_flask_manifest.head(10))


# ------------------------------------------------------------
# 10. Pipeline files summary
# ------------------------------------------------------------

pipeline_rows = []

for name, path in files.items():
    pipeline_rows.append({
        "file_key": name,
        "path": str(path),
        "exists": path.exists(),
        "folder": str(path.parent),
        "filename": path.name
    })

pipeline_rows.append({
    "file_key": "models_folder",
    "path": str(MODELS_DIR),
    "exists": MODELS_DIR.exists(),
    "folder": str(MODELS_DIR.parent),
    "filename": MODELS_DIR.name
})

pipeline_rows.append({
    "file_key": "predictions_folder",
    "path": str(PREDICTIONS_DIR),
    "exists": PREDICTIONS_DIR.exists(),
    "folder": str(PREDICTIONS_DIR.parent),
    "filename": PREDICTIONS_DIR.name
})

pipeline_rows.append({
    "file_key": "final_project_summary_folder",
    "path": str(FINAL_SUMMARY_DIR),
    "exists": FINAL_SUMMARY_DIR.exists(),
    "folder": str(FINAL_SUMMARY_DIR.parent),
    "filename": FINAL_SUMMARY_DIR.name
})

final_pipeline_files = pd.DataFrame(pipeline_rows)
final_pipeline_files.to_csv(final_pipeline_files_file, index=False)

print("\n✅ Pipeline files summary saved:")
print(final_pipeline_files_file)
display(final_pipeline_files)


# ------------------------------------------------------------
# 11. Create final project README text
# ------------------------------------------------------------

best_overall_model = "Unknown"
best_overall_r2 = np.nan
best_overall_rp = np.nan

if len(final_model_performance) > 0:
    best_overall_model = final_model_performance.iloc[0].get("model", "Unknown")
    best_overall_r2 = final_model_performance.iloc[0].get("mean_r2_score", np.nan)
    best_overall_rp = final_model_performance.iloc[0].get("mean_pearson_rp", np.nan)

readme_text = f"""
NCI ALMANAC Drug Combination ComboScore Prediction Project
==========================================================

PROJECT STATUS
--------------
Step 1  : FG-only ComboScore created
Step 2  : Drug features cleaned
Step 2B : Missing drug investigation completed
Step 2C : Missing drug features recovered
Step 3  : Model matrix created
Step 4  : One-cell-line model test completed
Step 5  : Official evaluation completed
Step 6  : Final 60 models trained on 100% data
Step 7A : Official test-set prediction checking completed
Step 7B : Single custom prediction completed
Step 7C : Batch custom prediction completed
Step 8  : Final summary files created


IMPORTANT FINAL DECISION
------------------------
Official evaluation metrics come from Step 5.

Step 6 final models are trained on 100% data for prediction/deployment.
Therefore Step 6 is not used for test metrics.

The failed optimized Step 6 experiment with interaction features was rejected because
it performed worse than the Step 5 baseline.


OFFICIAL BEST OVERALL MODEL
---------------------------
Best average model by Step 5 mean R2:
Model       : {best_overall_model}
Mean R2     : {best_overall_r2}
Mean Rp     : {best_overall_rp}


FINAL MODEL COUNTS
------------------
{final_model_counts.to_string(index=False)}


DATA SUMMARY
------------
Available drugs       : {len(available_drugs)}
Available cell lines  : {len(available_cell_lines)}
Final model features  : {len(final_feature_columns)}
Final saved models    : {len(final_model_files)}


FINAL PREDICTION FLOW
---------------------
Input:
    NSC1
    NSC2
    CELLNAME

Process:
    1. Load data/drug_features.csv
    2. Get 263 features for NSC1
    3. Get 263 features for NSC2
    4. Create 526 model features:
           D1_feat_0 ... D1_feat_262
           D2_feat_0 ... D2_feat_262
    5. Load correct model for CELLNAME using step6_final_model_registry.csv
    6. Predict NSC1 -> NSC2
    7. Predict NSC2 -> NSC1
    8. Average both predictions
    9. Return final predicted ComboScore


FILES NEEDED FOR FLASK APP
--------------------------
1. data/drug_features.csv
2. data/step6_final_model_feature_columns.json
3. results/step6_final_model_registry.csv
4. models/final_step6_*.pkl


IMPORTANT FOLDERS
-----------------
Project folder:
{PROJECT_DIR}

Data folder:
{DATA_DIR}

Results folder:
{RESULTS_DIR}

Models folder:
{MODELS_DIR}

Predictions folder:
{PREDICTIONS_DIR}

Final summary folder:
{FINAL_SUMMARY_DIR}


FINAL SUMMARY FILES CREATED
---------------------------
1. final_model_performance_summary.csv
2. final_best_models_per_cellline.csv
3. final_model_counts.csv
4. final_available_inputs.csv
5. final_flask_assets_manifest.csv
6. final_pipeline_files_summary.csv
7. final_project_readme.txt
8. available_cell_lines.txt
9. available_drug_nscs.txt


NOTE FOR CODEX / FLASK FRONTEND
-------------------------------
Use the Step 7B logic for single prediction in Flask.

The Flask app should:
    - Accept NSC1, NSC2, CELLNAME from user
    - Validate NSCs using drug_features.csv
    - Validate cell line using step6_final_model_registry.csv
    - Build the 526-feature input row
    - Load the correct final model
    - Predict both drug orders
    - Return the average predicted ComboScore

"""

with open(final_project_readme_file, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("\n✅ Final project README saved:")
print(final_project_readme_file)


# ------------------------------------------------------------
# 12. Final output display
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 8 FINAL PROJECT SUMMARY COMPLETE")
print("============================================================")

print("\nFinal summary folder:")
print(FINAL_SUMMARY_DIR)

print("\nCreated files:")
created_files = [
    final_model_performance_file,
    final_best_models_file,
    final_model_counts_file,
    final_available_inputs_file,
    final_flask_manifest_file,
    final_pipeline_files_file,
    final_project_readme_file,
    final_cell_lines_file,
    final_drugs_file
]

for file in created_files:
    print("✅", file)

print("\nUse this file first for Codex/Flask:")
print(final_project_readme_file)

print("\nUse this file to know which assets Flask needs:")
print(final_flask_manifest_file)

print("\n✅ STEP 8 COMPLETE")

✅ Final summary folder ready:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary

Checking important files:
combo_score                         -> FOUND
drug_features                       -> FOUND
model_matrix                        -> FOUND
feature_columns                     -> FOUND
step5_all_model_comparison          -> FOUND
step5_best_model_per_cellline       -> FOUND
step5_average_model_performance     -> FOUND
step6_final_model_summary           -> FOUND
step6_final_model_registry          -> FOUND
step7a_official_test_dataset        -> FOUND
step7a_best_model_test_predictions  -> FOUND
step7b_single_prediction            -> FOUND
step7c_batch_input                  -> FOUND
step7c_batch_output                 -> FOUND

✅ Main files loaded
Drug features shape: (100, 264)
Step 5 average performance shape: (4, 10)
Step 5 best models shape: (60, 17)
Step 6 final summary shape: (60, 14)
Step 6 registry shape: (60, 6)
Final feature count: 526

✅ Final model perfo

,model,mean_r2_score,median_r2_score,mean_rmse,mean_mae,mean_pearson_rp,median_pearson_rp,mean_spearman_rs,total_train_time_sec,successful_cellline_count
0,CatBoost,0.374818,0.354692,44.221958,31.615502,0.617905,0.616092,0.572928,299.560889,60
1,RandomForest,0.371234,0.366933,44.317994,31.315135,0.612695,0.614800,0.584765,660.304590,60
2,LightGBM,0.370570,0.372275,44.290690,31.475354,0.612983,0.618274,0.580449,177.991081,60
3,XGBoost,0.369660,0.355505,44.422604,31.646120,0.611927,0.602964,0.575614,153.826517,60



✅ Final best models per cell line saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_best_models_per_cellline.csv


,cell_line,model,rows_total,train_rows_before_augmentation,train_rows_after_augmentation,test_rows,input_features,r2_score,rmse,mae,pearson_rp,spearman_rs,train_time_sec,model_path_x,prediction_file,status,error,selected_model,model_path_y
0,786-0,CatBoost,2439,2195,4390,244,526,0.341898,30.885618,24.368193,0.585791,0.587811,5.408880,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,CatBoost,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
1,A498,CatBoost,2441,2196,4392,245,526,0.447089,33.008112,23.161057,0.673602,0.641828,5.162141,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,CatBoost,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
2,A549/ATCC,LightGBM,2439,2195,4390,244,526,0.395637,29.374134,21.848186,0.629480,0.584564,2.669145,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,LightGBM,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
3,ACHN,RandomForest,2434,2190,4380,244,526,0.371066,42.416528,31.872216,0.623041,0.564880,11.076158,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,RandomForest,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
4,BT-549,RandomForest,2431,2187,4374,244,526,0.336038,48.121570,33.925945,0.585430,0.542594,11.080332,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,RandomForest,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
5,CAKI-1,LightGBM,2436,2192,4384,244,526,0.378873,43.512602,30.896801,0.622691,0.597871,2.756468,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,LightGBM,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
6,CCRF-CEM,RandomForest,2399,2159,4318,240,526,0.386014,57.319395,42.582848,0.627295,0.513332,11.103961,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,RandomForest,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
7,COLO 205,CatBoost,2426,2183,4366,243,526,0.341847,47.043135,35.993181,0.588019,0.532671,4.895819,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,CatBoost,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
8,DU-145,XGBoost,2427,2184,4368,243,526,0.434345,44.379482,27.544412,0.662433,0.616230,2.573967,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,XGBoost,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...
9,EKVX,XGBoost,2439,2195,4390,244,526,0.513032,37.043191,26.724034,0.739068,0.649060,2.515709,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,success,NaN,XGBoost,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...



✅ Final model counts saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_model_counts.csv


...


✅ Flask assets manifest saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_flask_assets_manifest.csv


,asset_type,required_for_flask,path,description
0,drug_features,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,Contains 263 features for each available drug ...
1,feature_columns,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,Exact 526 feature column order used by final m...
2,model_registry,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,Maps each cell line to its selected final mode...
3,models_folder,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,Folder containing final trained .pkl models.
4,single_prediction_example,no,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,Example output from one custom prediction.
5,batch_prediction_example,no,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,Example output from batch prediction.
6,final_model_file,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,One final trained model file for a specific ce...
7,final_model_file,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,One final trained model file for a specific ce...
8,final_model_file,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,One final trained model file for a specific ce...
9,final_model_file,yes,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,One final trained model file for a specific ce...



✅ Pipeline files summary saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_pipeline_files_summary.csv


,file_key,path,exists,folder,filename
0,combo_score,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,comboscore.csv
1,drug_features,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,drug_features.csv
2,model_matrix,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,model_matrix.csv
3,feature_columns,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step6_final_model_feature_columns.json
4,step5_all_model_comparison,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step5_all_cellline_model_comparison.csv
5,step5_best_model_per_cellline,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step5_best_model_per_cellline.csv
6,step5_average_model_performance,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step5_average_model_performance.csv
7,step6_final_model_summary,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step6_final_model_summary.csv
8,step6_final_model_registry,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step6_final_model_registry.csv
9,step7a_official_test_dataset,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,True,C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v...,step5_official_test_dataset.csv



✅ Final project README saved:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_project_readme.txt

STEP 8 FINAL PROJECT SUMMARY COMPLETE

Final summary folder:
C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary

Created files:
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_model_performance_summary.csv
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_best_models_per_cellline.csv
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_model_counts.csv
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_available_inputs.csv
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_flask_assets_manifest.csv
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_pipeline_files_summary.csv
✅ C:\Users\HP\Desktop\SDP 27 april\nci_almanac_v3\final_project_summary\final_project_readme.txt
✅ C:\Users\